<style>
/* Estilo global para todo o notebook */
.jp-Notebook, .notebook {
    max-width: 1000px;
    margin: 0 auto;
}

div.text_cell_render {
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Arial, sans-serif;
    line-height: 1.6;
    color: #333;
}

/* Títulos */
h1, h2, h3, h4, h5, h6 {
    margin-top: 1.5em;
    margin-bottom: 0.5em;
    font-weight: 600;
    color: #2c6e9e;
}

h1 { font-size: 2em; border-bottom: 2px solid #2c6e9e; padding-bottom: 10px; }
h2 { font-size: 1.5em; border-bottom: 1px solid #ddd; padding-bottom: 5px; }
h3 { font-size: 1.3em; }

/* Parágrafos e listas */
p, li {
    text-align: justify;
}

/* Caixas de destaque */
.box-info {
    background-color: #e8f4fd;
    border-left: 5px solid #2196f3;
    padding: 15px;
    border-radius: 8px;
    margin: 15px 0;
}

.box-warning {
    background-color: #fff3cd;
    border-left: 5px solid #ff9800;
    padding: 15px;
    border-radius: 8px;
    margin: 15px 0;
}

.box-success {
    background-color: #e6f4ea;
    border-left: 5px solid #4caf50;
    padding: 15px;
    border-radius: 8px;
    margin: 15px 0;
}

/* Tabelas */
table {
    border-collapse: collapse;
    width: 100%;
    margin: 15px 0;
}

th, td {
    border: 1px solid #ddd;
    padding: 10px;
    text-align: left;
}

th {
    background-color: #f2f2f2;
    font-weight: 600;
}
</style>

<div style="max-width: 900px; margin: 0 auto; font-family: Arial, sans-serif; line-height: 1.6;">

<br>

<div align="center">
    <img src="
X9u0k/YOdr2lLWiu3/+ezLoWd1WMyO7FPDc4dtodGvMGwWMsRsbWmlFLWkPXR0XngnaOpr2fJe3xYonRFxTH/wHIBpxd2aZzVAAAAABJRU5ErkJggg" width="150"/>
</div>

<br>

<div align="center">
    <h1 style="font-size: 30px; font-weight: 900; margin: 0; padding: 0;">Projeto GEOTEC</h1>
</div>

<br>

<div align="center">
    <p style="font-size: 18px; margin-top: 10px; margin-bottom: 10px;">
        Capacitação em Geotecnologias para Monitoramento do Crédito Rural e do Proagro
    </p>
</div>

<br>

<div align="center">
    <p><b>Módulo:</b> Governança Territorial</p>
</div>

---

### Equipe

Pedro Coutinho, Gerd Sparovek, Alberto Barretto, Herbert Lincon, Marluce Scarabello, Pietro Gragnolati, Ana Beatriz Ferreira Macedo, Rodrigo Fernando Maule, João Marinho, João Gabriel Souza, Victor Hugo Bitu Patricio, Richard Torsiano

---

### Instituição
Instituto para Governança Territorial e Políticas Públicas (iGPP)

---

</div>

<hr style="border:1px solid #000000;">

# Aula 17 - Indicadores Ambientais

<hr style="border:1px solid #000000;">

### Importação dos Pacotes

A célula abaixo importa todas as bibliotecas necessárias para este notebook. Cada importação é acompanhada de um comentário explicando sua função.

In [1]:
# ============================================================================
# Manipulação de dados e ambiente
# ============================================================================
import os
import gc
import binascii
from dataclasses import dataclass
import dotenv
import random
import binascii

# Configurações de memória (ajuda a evitar travamentos)
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'

# ============================================================================
# Manipulação de dados tabulares e geoespaciais
# ============================================================================
import pandas as pd
import geopandas as gpd

# ============================================================================
# Conexão com banco de dados PostgreSQL/PostGIS
# ============================================================================
from sqlalchemy import create_engine, text
import urllib.parse

# ============================================================================
# Visualização
# ============================================================================
import matplotlib.pyplot as plt
import folium
import webbrowser
from IPython.display import display, Markdown, HTML

# ============================================================================
# Visualização cartográfica avançada
# ============================================================================
import cartopy.crs as ccrs
import cartopy.io.img_tiles as cimgt

# ============================================================================
# Manipulação de geometrias
# ============================================================================
from shapely import wkb
from shapely.geometry.base import BaseGeometry

print("Bibliotecas importadas com sucesso!")

Bibliotecas importadas com sucesso!


<hr style="border:1px solid #000000;">

# Conexão com o Banco de dados

<hr style="border:1px solid #000000;">

### Parâmetros de conexão com o banco de dados


Aqui vamos carregar as credenciais de conexão para o banco de dados. Os dados abaixos estão configurados de arcordo com a sugestão de banco de dados dado no curso. Caso queria acessar um outro banco, por favor, altere as informações abaixo de acordo com as suas credenciais no banco.

In [2]:
# ============================================================================
# PARÂMETROS DA ANÁLISE - ALTERE ESTES VALORES SE FOR NECESSARIO
# ============================================================================

# Credenciais de conexão com o banco (banco local do curso)
DB_USER = "postgres"           # Usuário padrão do PostgreSQL
DB_PASSWORD = "postgres"       # Senha configurada durante a instalação
DB_HOST = "localhost"          # Banco rodando no próprio computador
DB_PORT = "5432"               # Porta padrão do PostgreSQL
DB_NAME = "geotec"             # Nome do banco de dados criado para o curso

# Codifica a senha para evitar erros com caracteres especiais
encoded_password = urllib.parse.quote_plus(DB_PASSWORD)

# Monta a string de conexão
connection_string = f"postgresql+psycopg2://{DB_USER}:{encoded_password}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# Cria o motor de conexão
engine = create_engine(connection_string)

# Teste de conexão
try:
    with engine.connect() as conn:
        print(" Conectado com sucesso ao banco de dados!")
except Exception as e:
    print(" Erro ao conectar:")
    print(e)

 Conectado com sucesso ao banco de dados!


<hr style="border:1px solid #000000;">

#  1. Analisar e Estimar LPVN no Imóvel

<hr style="border:1px solid #000000;">

Traz de forma resumida as principais estimativas no imóvel, como: Déficit de APP, Déficit de RL e Excedente de Vegetação Nativa.

In [5]:
# ============================================================================
# Dados SICAR (Base de Dados do CAR)
# ============================================================================

CAR_SCHEMA = 'car'
SICAR_TABLE = 'es_pi_area_imovel_sicar'        		# Polígono principal do imóvel
APP_TABLE = 'es_pi_apps_sicar'                 		# Áreas de Preservação Permanente
VEGETACAO_TABLE = 'es_pi_vegetacao_nativa_sicar' 	# Remanescente de vegetação nativa
RESERVA_TABLE = 'es_pi_reserva_legal_sicar'    		# Reserva Legal declarada

# ============================================================================
# Dados INCRA (Módulos Fiscais)
# ============================================================================

INCRA_SCHEMA = 'incra'
MODULOS_FISCAIS_TABLE = 'pa_br_modulosfiscais_incra' # Parâmetros de módulo fiscal

# ============================================================================
# Dados IBGE (Território e Limites Legais)
# ============================================================================

IBGE_SCHEMA = 'ibge'
AMAZONIA_LEGAL_TABLE = 'pa_br_amazonia_legal'  		# Delimitação da Amazônia e Regiões
MUNICIPIOS_TABLE = 'es_pi_municipios_ibge_2025' 	# Malha municipal (PI)

# ============================================================================
# SELECIONE O CODIGO CAR DO IMOVEL QUE DESEJA ANALISAR
# ============================================================================

COD_IMOVEL = 'PI-2200400-A377B1177E914D8792BCE4AB99F39875' 	 # Possui todas as feições
# COD_IMOVEL = 'PI-2204303-C537B8423457434386A8003BB50CF0E4' # sem VN declarada
# COD_IMOVEL = 'PI-2204303-59FFAE72C88648FAA013FAE36A658BA0' # sem APP
# COD_IMOVEL = 'PI-2210607-76119AF3DBCB499097E9772D950C48D4' # Possui déficit APP
# COD_IMOVEL = 'PI-2211209-B3866F03F500495F9EFC79FF754E6A1D' # Possui todos os déficits
# COD_IMOVEL = 'PI-2210102-299FE936E65A4FBE8BADF2C526B91D52' # Possui todos os déficits e excedente VN
# COD_IMOVEL = 'PI-2202802-1B5D000BE5C5446998AFB37199F74C71' # Possui excedente VN

# ============================================================================
# CONSULTA SQL PARA ANÁLISE DO IMÓVEL SELECIONADO
# ============================================================================

query_car = text(f"""WITH car_analisar AS (
	SELECT * FROM {CAR_SCHEMA}.{SICAR_TABLE} WHERE cod_imovel = '{COD_IMOVEL}'
),
imovel AS (
	SELECT
		a.cod_imovel,
		a.cod_estado,
		CASE 
			WHEN b.regiao IS NULL THEN 'Fora da Amazônia Legal'
			ELSE b.regiao
		END AS regiao,
		((st_area(st_transform(st_intersection(a.geom, b.geom),97823))/10000) /
			(st_area(st_transform(a.geom,97823))/10000)*100)
			AS porcentagem_do_imovel,
		st_area(st_transform(a.geom,97823))/10000 AS area_imovel_ha
	FROM car_analisar a
	LEFT JOIN {IBGE_SCHEMA}.{AMAZONIA_LEGAL_TABLE} b
		ON st_intersects(a.geom, b.geom)
),
app AS (	
	SELECT
		a.cod_imovel, 
		st_union(d.geom) AS app_declarada,
		CASE 
			WHEN st_area(st_transform(a.geom,97823))/10000 / c.modulofiscal_ha <= 4 THEN 'Sim'
			ELSE 'Não (Depende do ZEE)'
		END
		AS apto_para_61a,
		st_union(d.geom) 
			FILTER (WHERE d.cod_tema LIKE 'APP_ESCADINHA%') AS app_minima,
		st_union(e.geom) AS remanescente_vegetacao
	FROM car_analisar a
	JOIN {IBGE_SCHEMA}.{MUNICIPIOS_TABLE} b
		ON ST_Intersects(a.geom, b.geom)
	JOIN {INCRA_SCHEMA}.{MODULOS_FISCAIS_TABLE} c
		ON b.cd_mun::int = c.codibge 
	LEFT JOIN {CAR_SCHEMA}.{APP_TABLE} d
		USING (cod_imovel)
	LEFT JOIN {CAR_SCHEMA}.{VEGETACAO_TABLE} e
		USING (cod_imovel)
	GROUP BY 
		a.cod_imovel, a.geom, c.modulofiscal_ha, d.cod_imovel
), 
calc_app AS (
	SELECT 
		*,
		st_area(st_transform(app_declarada,97823))/10000 AS app_declarada_ha,
		st_area(st_transform(app_minima,97823))/10000 AS app_minima_ha,
		COALESCE(
				st_area(st_transform(st_difference(
					app_declarada , remanescente_vegetacao
					),97823))/10000,
				st_area(st_transform(app_declarada,97823))/10000
				) AS deficit_app_total_ha,
		COALESCE(
				st_area(st_transform(st_difference(
					app_minima , remanescente_vegetacao
					),97823))/10000,
				st_area(st_transform(app_minima,97823))/10000
				) AS deficit_app_minima_ha
	FROM app
),
rl_exigida AS (
	SELECT 
		*,
		CASE
			WHEN regiao = 'Fora da Amazônia Legal' THEN 
				(area_imovel_ha*(porcentagem_do_imovel/100))*0.2
			WHEN regiao = 'Cerrado' THEN
				CASE
					WHEN cod_estado = 'PI' THEN 
						(area_imovel_ha*(porcentagem_do_imovel/100))*0.3
					ELSE 
						(area_imovel_ha*(porcentagem_do_imovel/100))*0.2
				END
			WHEN regiao = 'Amazônia Legal' THEN 
				(area_imovel_ha*(porcentagem_do_imovel/100))*0.8
			WHEN regiao = 'Cerrado na Amazônia Legal' THEN 
				COALESCE((area_imovel_ha*(porcentagem_do_imovel/100))*0.35, area_imovel_ha*0.35)
		END AS rl_exigida_ha
	FROM imovel
),
rl_declarada AS (
	SELECT 
		a.cod_imovel, 
		CASE 
			WHEN b.regiao IS NULL THEN 'Fora da Amazônia Legal'
			ELSE b.regiao
		END AS regiao,
		st_union(st_intersection(a.geom,b.geom)) AS rl
	FROM {CAR_SCHEMA}.{RESERVA_TABLE} a
	LEFT JOIN {IBGE_SCHEMA}.{AMAZONIA_LEGAL_TABLE} b
		ON st_intersects(a.geom, b.geom)
	LEFT JOIN car_analisar c 
		USING (cod_imovel)
	WHERE 
		c.cod_imovel IS NOT NULL
	GROUP BY 
		a.cod_imovel, a.geom, b.geom, b.regiao
),
vn_declarada AS (
	SELECT 
		a.cod_imovel, 
		CASE 
			WHEN b.regiao IS NULL THEN 'Fora da Amazônia Legal'
			ELSE b.regiao
		END AS regiao,
		st_union(st_intersection(a.geom,b.rl)) AS vn,
		(st_area(st_transform(st_intersection(a.geom,b.rl),97823))/10000) AS area_vn_rl_ha
	FROM {CAR_SCHEMA}.{VEGETACAO_TABLE} a
	LEFT JOIN rl_declarada b
			ON st_intersects(a.geom, b.rl)
	LEFT JOIN car_analisar c 
		ON a.cod_imovel = c.cod_imovel
	WHERE 
		c.cod_imovel IS NOT NULL
	GROUP BY 
		a.cod_imovel, a.geom, b.rl, b.regiao
),
dados_finais AS (
	SELECT 
		a.cod_imovel, 
		st_union(b.rl) AS rl_declarada,
		apto_para_61a,
		sum(rl_exigida_ha) AS rl_exigida_ha,
		sum(CASE
			WHEN c.area_vn_rl_ha - rl_exigida_ha < 0 THEN rl_exigida_ha - c.area_vn_rl_ha
			WHEN c.area_vn_rl_ha - rl_exigida_ha >= 0 THEN 0
			ELSE rl_exigida_ha
		END) AS deficit_rl_ha,
		d.deficit_app_total_ha, 
		d.deficit_app_minima_ha,
		d.app_declarada_ha, d.app_minima_ha,
		st_area(st_transform(
			COALESCE(st_difference(COALESCE(st_difference(e.geom, st_union(d.app_declarada)), e.geom), st_union(rl)), e.geom)
			,97823))/10000 AS excedente_ha,
		st_union(d.app_minima) AS app,
		st_union(d.app_declarada) AS app_total,
		st_union(e.geom) AS vn_declarada
	FROM rl_exigida a
	LEFT JOIN rl_declarada b 
		USING (cod_imovel, regiao)
	LEFT JOIN vn_declarada c
		USING (cod_imovel, regiao)
	LEFT JOIN calc_app d 
		USING (cod_imovel)
	LEFT JOIN {CAR_SCHEMA}.{VEGETACAO_TABLE} e 
		USING (cod_imovel)
	GROUP BY 
		a.cod_imovel, apto_para_61a, d.app_declarada_ha, d.app_minima_ha,
		d.deficit_app_total_ha, d.deficit_app_minima_ha, e.geom
)
SELECT
    cod_imovel,
    -- 1. Indicadores de Déficit (Comparação de áreas)
    CASE 
        WHEN deficit_rl_ha > 0 THEN 'Sim'
        ELSE 'Não'
    END AS possui_deficit_rl,
    CASE 
        WHEN deficit_app_total_ha > 0 THEN 'Sim'
        ELSE 'Não'
    END AS possui_deficit_app_total,
    CASE 
        WHEN deficit_app_minima_ha > 0 THEN 'Sim'
        ELSE 'Não'
    END AS possui_deficit_app_minima,
    -- 2. Regras de Transição (Art. 61-A)
    apto_para_61a,
    -- 3. Análise de Excedentes e Proporções de Cobertura
    excedente_ha AS vn_fora_app_fora_rl_ha,
    CASE 
        WHEN excedente_ha > 0 THEN 'Sim'
        ELSE 'Não'
    END AS possui_excedente_vn,
    deficit_rl_ha,
    -- Cálculo de percentual de áreas sem vegetação nativa
    CASE 
        WHEN rl_exigida_ha > 0 THEN (deficit_rl_ha / rl_exigida_ha) * 100
        ELSE 0
    END AS perc_rl_sem_vn,
    deficit_app_total_ha,
    CASE 
        WHEN app_declarada_ha > 0 THEN (deficit_app_total_ha / app_declarada_ha) * 100
        ELSE 0
    END AS perc_app_total_sem_vn,
    deficit_app_minima_ha,
    CASE 
        WHEN app_minima_ha > 0 THEN (deficit_app_minima_ha / app_minima_ha) * 100
        ELSE 0
    END AS perc_app_minima_sem_vn, 
    -- 4. Camadas de Geometrias Completas (Shapely/Folium)
    b.geom AS imovel,
    rl_declarada,
    app_total AS app_total_declarada,
    app AS app_minima,
    vn_declarada
FROM dados_finais
LEFT JOIN car_analisar b 
    USING (cod_imovel);""")

df_car = pd.read_sql_query(query_car, engine)
df = df_car.drop(columns=['imovel', 'rl_declarada', 'app_total_declarada', 'app_minima', 'vn_declarada'])

# 1. Definição do Dicionário de Metadados
metadados_dict = {
    "Campo": [
        "cod_imovel", "possui_deficit_rl", "possui_deficit_app_total", 
        "possui_deficit_app_minima", "apto_para_61a", "vn_fora_app_fora_rl_ha",
        "possui_excedente_vn", "deficit_rl_ha", "perc_rl_sem_vn", 
        "deficit_app_total_ha", "perc_app_total_sem_vn", "deficit_app_minima_ha",
        "perc_app_minima_sem_vn", "imovel", "rl_declarada", 
        "app_total_declarada", "app_minima", "vn_declarada"
    ],
    "Descrição": [
        "Código do Imóvel no sistema", "Indicador se possui déficit em Reserva Legal",
        "Indicador se possui déficit em APP Total", "Indicador se possui déficit em APP mínima",
        "Indicador se imóvel está apto a ter reduções segundo o artigo 61-A",
        "Área em hectares de Vegetação Nativa fora de APP e fora de Reserva Legal",
        "Indicador se possui Excedente de Vegetação Nativa", "Área em hectares do déficit em Reserva Legal",
        "Porcentagem da Reserva Legal sem Vegetação Nativa", "Área em hectares do déficit em APP Total",
        "Porcentagem da APP Total sem Vegetação Nativa", "Área em hectares do déficit em APP Mínima",
        "Porcentagem da APP Mínima sem Vegetação Nativa", "Geometria do imóvel",
        "Geometria da Reserva Legal declarada", "Geometria da APP Total declarada",
        "Geometria da APP Mínima declarada", "Geometria do Remanescente de Vegetação Nativa declarado"
    ]
}

df_metadados = pd.DataFrame(metadados_dict)

# --- 1. CONFIGURAÇÃO DE ESTILO ---
estilo_centralizado = [
    {'selector': 'th', 'props': [('text-align', 'center')]}, # Cabeçalhos
    {'selector': 'td', 'props': [('text-align', 'center')]}  # Células
]
# --- 2. EXIBIÇÃO DO RESULTADO ---
print("--- RESULTADO DA ANÁLISE ---")
display(df.head().style.set_table_styles(estilo_centralizado).hide(axis='index'))

print("\n--- DICIONÁRIO DE METADADOS ---")
display(df_metadados.style.set_table_styles(estilo_centralizado).hide(axis='index'))

--- RESULTADO DA ANÁLISE ---


cod_imovel,possui_deficit_rl,possui_deficit_app_total,possui_deficit_app_minima,apto_para_61a,vn_fora_app_fora_rl_ha,possui_excedente_vn,deficit_rl_ha,perc_rl_sem_vn,deficit_app_total_ha,perc_app_total_sem_vn,deficit_app_minima_ha,perc_app_minima_sem_vn
PI-2200400-A377B1177E914D8792BCE4AB99F39875,Sim,Sim,Sim,Não (Depende do ZEE),453.556441,Sim,139.295117,34.854420,8.052601,20.666107,2.880871,100.000000



--- DICIONÁRIO DE METADADOS ---


Campo,Descrição
cod_imovel,Código do Imóvel no sistema
possui_deficit_rl,Indicador se possui déficit em Reserva Legal
possui_deficit_app_total,Indicador se possui déficit em APP Total
possui_deficit_app_minima,Indicador se possui déficit em APP mínima
apto_para_61a,Indicador se imóvel está apto a ter reduções segundo o artigo 61-A
vn_fora_app_fora_rl_ha,Área em hectares de Vegetação Nativa fora de APP e fora de Reserva Legal
possui_excedente_vn,Indicador se possui Excedente de Vegetação Nativa
deficit_rl_ha,Área em hectares do déficit em Reserva Legal
perc_rl_sem_vn,Porcentagem da Reserva Legal sem Vegetação Nativa
deficit_app_total_ha,Área em hectares do déficit em APP Total


<hr style="border:1px solid #000000;">

# Visualizar Imóvel e Estimativas LPVN

<hr style="border:1px solid #000000;">

Gera um arquivo HTML com a visualização do imóvel os locais onde existem Déficits de APP, Reserva Legal e os Remanescentes de Vegetação Nativa declarados.

In [ ]:
# ============================================================================
# Processamento Geográfico e Visualização Multi-Camadas 
# ============================================================================

# Função para converter WKB do PostGIS para Geometria Shapely
def converter_wkb(wkb_data):
    if wkb_data is None:
        return None
    try:
        # Se for string/hex do PostGIS
        if isinstance(wkb_data, str):
            return wkb.loads(binascii.unhexlify(wkb_data))
        # Se já for bytes
        return wkb.loads(wkb_data)
    except:
        return None

# Lista das colunas de geometria que sua consulta retorna
colunas_geom = ['imovel', 'rl_declarada', 'app_total_declarada', 'app_minima', 'vn_declarada']

# Converter colunas de geometria de forma segura (Idempotente)
for col in colunas_geom:
    if col in df_car.columns:
        # Busca o primeiro valor não nulo para verificar o tipo
        dados_validos = df_car[col].dropna()
        primeiro_valor = dados_validos.iloc[0] if not dados_validos.empty else None
        
        # Só converte se o dado existir e não for um objeto geométrico do Shapely
        if primeiro_valor is not None and not isinstance(primeiro_valor, BaseGeometry):
            df_car[col] = df_car[col].apply(converter_wkb)

# 1. Criar um GeoDataFrame base (usando o limite do imóvel para o centroide)
gdf_base = gpd.GeoDataFrame(df_car, geometry='imovel', crs="EPSG:4326")

# 2. Calcular o centroide para focar o mapa
centroid_geom = gdf_base.to_crs(epsg=5880).geometry.centroid.to_crs(epsg=4326).iloc[0]

# 3. Inicializar o mapa Folium com suporte a super zoom (não deixa o fundo cinza)
m = folium.Map(location=[centroid_geom.y, centroid_geom.x], zoom_start=14, max_zoom=22, control_scale=True)

# 4. Camadas de Fundo (Satélite com limitador nativo)
folium.TileLayer(
    'https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}', 
    attr='Google', 
    name='Google Satellite',
    max_native_zoom=18,  # Evita o erro 404 do servidor de imagens
    max_zoom=22         # Permite o Leaflet esticar o pixel para análise fina de vetores
).add_to(m)

# 5. Mapeamento de Atributos para o Tooltip (Dados do DataFrame)
colunas_atributos = [
    'cod_imovel', 'possui_deficit_rl', 'possui_deficit_app_total', 
    'possui_deficit_app_minima', 'apto_para_61a', 'vn_fora_app_fora_rl_ha',
    'possui_excedente_vn', 'deficit_rl_ha', 'perc_rl_sem_vn', 
    'deficit_app_total_ha', 'perc_app_total_sem_vn', 'deficit_app_minima_ha',
    'perc_app_minima_sem_vn'
]

# Filtra apenas o que de fato veio na query do df_car para não dar KeyError
colunas_validas = [col for col in colunas_atributos if col in df_car.columns]

# Dicionário de tradução dos nomes das colunas na caixinha do mouse
aliases_mapeamento = {
    'cod_imovel': 'Código Imóvel:',
    'possui_deficit_rl': 'Possui Déficit RL?:',
    'possui_deficit_app_total': 'Possui Déficit APP Total?:',
    'possui_deficit_app_minima': 'Possui Déficit APP Mínima?:',
    'apto_para_61a': 'Apto Art. 61-A?:',
    'vn_fora_app_fora_rl_ha': 'VN Fora RL (ha):',
    'possui_excedente_vn': 'Possui Excedente VN?:',
    'deficit_rl_ha': 'Déficit RL (ha):',
    'perc_rl_sem_vn': 'Perc RL sem VN (%):',
    'deficit_app_total_ha': 'Déficit APP Total (ha):',
    'perc_app_total_sem_vn': 'Perc APP Total sem VN (%):',
    'deficit_app_minima_ha': 'Déficit APP Mínima (ha):',
    'perc_app_minima_sem_vn': 'Perc APP Mínima sem VN (%):'
}
aliases_validos = [aliases_mapeamento[col] for col in colunas_validas]

# 6. Dicionário de Estilização para cada camada vetorial
camadas_config = {
    'imovel': {'nome': 'Limite do Imóvel', 'cor': 'red', 'fill': False, 'weight': 3},
    'vn_declarada': {'nome': 'Vegetação Nativa', 'cor': '#228B22', 'fill': True, 'opacity': 0.4},
    'rl_declarada': {'nome': 'Reserva Legal', 'cor': '#006400', 'fill': True, 'opacity': 0.6},
    'app_total_declarada': {'nome': 'APP Total', 'cor': '#4169E1', 'fill': True, 'opacity': 0.5},
    'app_minima': {'nome': 'APP Recomposição (Escadinha)', 'cor': '#FF4500', 'fill': True, 'opacity': 0.7}
}

# 7. Adicionar cada geometria ao mapa como uma camada separada com os dados tabulares associados
for col, config in camadas_config.items():
    if col in df_car.columns and df_car[col].iloc[0] is not None:
        
        # Unimos os dados textuais/numéricos à geometria ativa do loop
        temp_gdf = gpd.GeoDataFrame(df_car[colunas_validas + [col]], geometry=col, crs="EPSG:4326")
        
        # Construção do Tooltip flutuante personalizado
        tooltip_camada = folium.GeoJsonTooltip(
            fields=colunas_validas,
            aliases=aliases_validos,
            localize=True,
            sticky=True,
            labels=True,
            style="""
                background-color: #F8F9FA;
                border: 2px solid #2B2D42;
                border-radius: 4px;
                box-shadow: 4px 4px rgba(0,0,0,0.15);
                color: #2B2D42;
                font-family: 'Courier New', monospace;
                font-size: 12px;
                padding: 12px;
            """
        )
        
        folium.GeoJson(
            temp_gdf,
            name=config['nome'],
            style_function=lambda x, c=config: {
                'fillColor': c['cor'] if c['fill'] else 'transparent',
                'color': c['cor'],
                'weight': c.get('weight', 2),
                'fillOpacity': c.get('opacity', 0.5)
            },
            tooltip=tooltip_camada
        ).add_to(m)

# 8. Adicionar controles e salvar
folium.LayerControl().add_to(m)

nome_arquivo = 'imovel_areas_protegidas.html'
m.save(nome_arquivo)

webbrowser.open(nome_arquivo)

True

<hr style="border:1px solid #000000;">

# 2. Carregar Feições Declaradas no Imóvel

<hr style="border:1px solid #000000;">

Carrega todas os tipos de feições declaradas no imóvel.

In [7]:
# ============================================================================
# Dados SICAR - Tabelas de Feições do Imóvel
# ============================================================================

CAR_SCHEMA = 'car'

SICAR_TABLE = 'es_pi_area_imovel_sicar'                # Polígono principal
APP_TABLE = 'es_pi_apps_sicar'                         # APPs
RESERVA_TABLE = 'es_pi_reserva_legal_sicar'            # Reserva Legal
HIDROGRAFIA_TABLE = 'es_pi_hidrografia_sicar'          # Corpos d'água
USO_RESTRITO_TABLE = 'es_pi_uso_restrito_sicar'        # Áreas de uso restrito
CONSOLIDADA_TABLE = 'es_pi_area_consolidada_sicar'     # Área consolidada
POUSIO_TABLE = 'es_pi_area_pousio_sicar'               # Área de pousio
SERVIDAO_TABLE = 'es_pi_servidao_administrativa_sicar' # Servidão administrativa
VEGETACAO_TABLE = 'es_pi_vegetacao_nativa_sicar'       # Vegetação nativa

# ============================================================================
# SELECIONE O CODIGO CAR
# ============================================================================

COD_IMOVEL = 'PI-2206209-137582EF6FAC4409A69D049A8C1447B0'

# ============================================================================
# CONSULTA SQL PARA ANÁLISE DO IMÓVEL SELECIONADO
# ============================================================================

query_feicoes = text(f"""
WITH car_analisar AS (
    SELECT * FROM {CAR_SCHEMA}.{SICAR_TABLE}
    WHERE cod_imovel = '{COD_IMOVEL}'
)
SELECT
    a.cod_imovel, 
    a.ind_tipo, 
    a.geom AS imovel,
    st_union(b.geom) AS app, 
    st_union(c.geom) AS rl,
    st_union(d.geom) AS hidrografia, 
    st_union(e.geom) AS uso_restrito,
    st_union(f.geom) AS consolidado, 
    st_union(g.geom) AS pousio,
    st_union(h.geom) AS servidao, 
    st_union(i.geom) AS vn
FROM car_analisar a
LEFT JOIN {CAR_SCHEMA}.{APP_TABLE} b
    USING (cod_imovel)
LEFT JOIN {CAR_SCHEMA}.{RESERVA_TABLE} c
    USING (cod_imovel)
LEFT JOIN {CAR_SCHEMA}.{HIDROGRAFIA_TABLE} d
    USING (cod_imovel)
LEFT JOIN {CAR_SCHEMA}.{USO_RESTRITO_TABLE} e
    USING (cod_imovel)
LEFT JOIN {CAR_SCHEMA}.{CONSOLIDADA_TABLE} f
    USING (cod_imovel)
LEFT JOIN {CAR_SCHEMA}.{POUSIO_TABLE} g
    USING (cod_imovel)
LEFT JOIN {CAR_SCHEMA}.{SERVIDAO_TABLE} h
    USING (cod_imovel)
LEFT JOIN {CAR_SCHEMA}.{VEGETACAO_TABLE} i
    USING (cod_imovel)
GROUP BY
    a.cod_imovel, a.ind_tipo, a.geom;
""")

df_feicoes = pd.read_sql_query(query_feicoes, engine)
df = df_feicoes[['cod_imovel', 'ind_tipo']]  # Seleciona apenas as colunas de interesse
 
# ============================================================================
# Definição do Dicionário de Metadados 
# ============================================================================

metadados_geometrias_dict = {
    "Campo": [
        "cod_imovel", "ind_tipo", "imovel", "app", "rl", 
        "hidrografia", "uso_restrito", "consolidado", 
        "pousio", "servidao", "vn"
    ],
    "Descrição": [
        "Código do Imóvel no sistema",
        "Indicador do tipo de imóvel declarado (IRU, AST, PCT)",
        "Geometria do imóvel",
        "Geometria da APP Total declarada",
        "Geometria da Reserva Legal declarada",
        "Geometria da Hidrografia declarada",
        "Geometria do Uso Restrito declarado",
        "Geometria da Área Consolidada declarada",
        "Geometria da Área de Pousio declarada",
        "Geometria da Servidão Administrativa declarada",
        "Geometria do Remanescente de Vegetação Nativa declarado"
    ]
}

df_meta_geom = pd.DataFrame(metadados_geometrias_dict)

# --- CONFIGURAÇÃO DE ESTILO ---
estilo_centralizado = [
    {'selector': 'th', 'props': [('text-align', 'center')]}, 
    {'selector': 'td', 'props': [('text-align', 'center')]}  
]

# --- EXIBIÇÃO ---
print("--- RESULTADO DA ANÁLISE ---")
display(df.head().style.set_table_styles(estilo_centralizado).hide(axis='index'))

print("\n--- DICIONÁRIO DE METADADOS ---")
display(df_meta_geom.style.set_table_styles(estilo_centralizado).hide(axis='index'))

--- RESULTADO DA ANÁLISE ---


cod_imovel,ind_tipo
PI-2206209-137582EF6FAC4409A69D049A8C1447B0,IRU



--- DICIONÁRIO DE METADADOS ---


Campo,Descrição
cod_imovel,Código do Imóvel no sistema
ind_tipo,"Indicador do tipo de imóvel declarado (IRU, AST, PCT)"
imovel,Geometria do imóvel
app,Geometria da APP Total declarada
rl,Geometria da Reserva Legal declarada
hidrografia,Geometria da Hidrografia declarada
uso_restrito,Geometria do Uso Restrito declarado
consolidado,Geometria da Área Consolidada declarada
pousio,Geometria da Área de Pousio declarada
servidao,Geometria da Servidão Administrativa declarada


<hr style="border:1px solid #000000;">

# Visualizar Todas as Feições Declaradas no Imóvel

<hr style="border:1px solid #000000;">

Gera um arquivo HTML com a visualização do imóvel e suas feições declaradas.

In [ ]:
# ============================================================================
# Processamento Geográfico e Visualização - Todas as Feições (df_feicoes)
# ============================================================================

# Função para converter WKB do PostGIS para Geometria Shapely
def converter_wkb(wkb_data):
    if wkb_data is None:
        return None
    try:
        # Se for string/hex do PostGIS
        if isinstance(wkb_data, str):
            return wkb.loads(binascii.unhexlify(wkb_data))
        # Se já for bytes
        return wkb.loads(wkb_data)
    except:
        return None

# 1. Lista expandida das colunas de geometria do df_feicoes
colunas_geom_feicoes = [
    'imovel', 'app', 'rl', 'hidrografia', 'uso_restrito', 
    'consolidado', 'pousio', 'servidao', 'vn'
]

# Converter colunas de geometria de forma segura (Idempotente)
for col in colunas_geom_feicoes:
    if col in df_feicoes.columns:
        # Busca o primeiro valor não nulo para verificar o tipo
        dados_validos = df_feicoes[col].dropna()
        primeiro_valor = dados_validos.iloc[0] if not dados_validos.empty else None
        
        # Só converte se o dado existir e ainda não for um objeto do Shapely
        if primeiro_valor is not None and not isinstance(primeiro_valor, BaseGeometry):
            df_feicoes[col] = df_feicoes[col].apply(converter_wkb)

# Criar GeoDataFrame base para o centroide
gdf_base_f = gpd.GeoDataFrame(df_feicoes, geometry='imovel', crs="EPSG:4326")
centroid_f = gdf_base_f.to_crs(epsg=5880).geometry.centroid.to_crs(epsg=4326).iloc[0]

# ============================================================================
# 2. Inicializar o mapa Folium com super zoom e suporte a camadas vetoriais complexas
# ============================================================================
m_f = folium.Map(location=[centroid_f.y, centroid_f.x], zoom_start=15, max_zoom=22, control_scale=True)

# 3. Camada de Satélite com limitador nativo do servidor
folium.TileLayer(
    'https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}', 
    attr='Google', 
    name='Google Satellite',
    max_native_zoom=18,  # Limite físico do Google
    max_zoom=22          # Permite esticar o pixel para conferência de limites
).add_to(m_f)

# ============================================================================
# 4. Configuração Dinamica de Atributos para Tooltip (todas as colunas exceto as de geometria)
# ============================================================================
# Seleciona absolutamente tudo o que não estiver mapeado na lista de geometrias
colunas_validas_f = [col for col in df_feicoes.columns if col not in colunas_geom_feicoes]

# Gera os rótulos de forma limpa automaticamente (Ex: 'ind_tipo' vira 'Ind Tipo:')
aliases_validos_f = [f"{col.replace('_', ' ').title()}:" for col in colunas_validas_f]

# 5. Dicionário de Estilização Completo para df_feicoes
config_feicoes = {
    'imovel':       {'nome': 'Limite do Imóvel', 'cor': 'white',   'fill': False, 'weight': 4},
    'vn':           {'nome': 'Vegetação Nativa', 'cor': '#228B22', 'fill': True,  'opacity': 0.4},
    'rl':           {'nome': 'Reserva Legal',    'cor': '#006400', 'fill': True,  'opacity': 0.5},
    'app':          {'nome': 'APP',              'cor': '#4169E1', 'fill': True,  'opacity': 0.5},
    'hidrografia':  {'nome': 'Hidrografia',      'cor': '#00FFFF', 'fill': True,  'opacity': 0.8},
    'uso_restrito': {'nome': 'Uso Restrito',     'cor': '#8B4513', 'fill': True,  'opacity': 0.5},
    'consolidado':  {'nome': 'Área Consolidada', 'cor': '#FFD700', 'fill': True,  'opacity': 0.4},
    'pousio':       {'nome': 'Área de Pousio',   'cor': '#DEB887', 'fill': True,  'opacity': 0.4},
    'servidao':     {'nome': 'Servidão Adm.',    'cor': '#800080', 'fill': True,  'opacity': 0.4}
}

# 6. Adicionar cada feição ao mapa como uma camada com dados atrelados
for col, config in config_feicoes.items():
    if col in df_feicoes.columns and df_feicoes[col].iloc[0] is not None:
        
        # Injeta dinamicamente todas as colunas de texto/números na geometria ativa da iteração
        temp_gdf = gpd.GeoDataFrame(df_feicoes[colunas_validas_f + [col]], geometry=col, crs="EPSG:4326")
        
        # Configuração do Tooltip Flutuante Dinâmico
        tooltip_feicao = folium.GeoJsonTooltip(
            fields=colunas_validas_f,
            aliases=aliases_validos_f,
            localize=True,
            sticky=True,
            labels=True,
            style="""
                background-color: #F8F9FA;
                border: 2px solid #1A1A1A;
                border-radius: 4px;
                box-shadow: 4px 4px rgba(0,0,0,0.15);
                color: #1A1A1A;
                font-family: 'Courier New', monospace;
                font-size: 12px;
                padding: 12px;
                white-space: nowrap;
            """
        )
        
        folium.GeoJson(
            temp_gdf,
            name=config['nome'],
            style_function=lambda x, c=config: {
                'fillColor': c['cor'] if c['fill'] else 'transparent',
                'color': c['cor'],
                'weight': c.get('weight', 2),
                'fillOpacity': c.get('opacity', 0.5)
            },
            tooltip=tooltip_feicao
        ).add_to(m_f)

# 7. Adicionar controles e salvar
folium.LayerControl().add_to(m_f)

nome_arquivo_feicoes = 'imovel_areas_declaradas.html'
m_f.save(nome_arquivo_feicoes)

webbrowser.open(nome_arquivo_feicoes)

True

<hr style="border:1px solid #000000;">

# 3. Analisar Informações do Imóvel

<hr style="border:1px solid #000000;">

Função que analisa alguns dados declarados no imóvel e realiza conferências de forma espacial.

In [10]:
# ============================================================================
# Definição do Schema CAR e Tabelas para Análise Tabular
# ============================================================================

CAR_SCHEMA = 'car'
SICAR_TABLE = 'es_pi_area_imovel_sicar'

# ============================================================================
# Definição do Schema IBGE e Tabelas para Análise Tabular
# ============================================================================

IBGE_SCHEMA = 'ibge'
MUNICIPIOS_TABLE = 'es_pi_municipios_ibge_2025'
AMAZONIA_LEGAL_TABLE = 'pa_br_amazonia_legal'

# ============================================================================
# Definição do Schema INCRA e Tabelas para Análise Tabular
# ============================================================================

INCRA_SCHEMA = 'incra'
MODULOS_FISCAIS_TABLE = 'pa_br_modulosfiscais_incra'

# ============================================================================
# Identificadores (COD_IMOVEL)
# ============================================================================

COD_IMOVEL = 'PI-2210102-299FE936E65A4FBE8BADF2C526B91D52' # Duas regiões
# COD_IMOVEL = 'PI-2200053-149E8DB10F0248B7A18BD20D3A1119B3' # Suspenso
# COD_IMOVEL = 'PI-2200400-A377B1177E914D8792BCE4AB99F39875' # Tamanho incorreto
# COD_IMOVEL = 'PI-2200053-02E4F2ACE1634A1ABAF119D95444EFED' # Município incorreto

# ============================================================================
# CONSULTA SQL PARA ANÁLISE DO IMÓVEL SELECIONADO
# ============================================================================

query_auditoria = text(f"""
WITH car_analisar AS (
    SELECT * FROM {CAR_SCHEMA}.{SICAR_TABLE}
    WHERE cod_imovel = '{COD_IMOVEL}'
),
municipio AS (
    SELECT 
        DISTINCT ON (a.cod_imovel)
        a.cod_imovel, b.nm_mun, b.cd_mun,
        CASE
            WHEN a.ind_status = 'SU' THEN 'Suspenso - Necessário Regularizar'
            WHEN a.ind_status = 'CA' THEN 'Cancelado - Necessário Regularizar'
            WHEN a.ind_status = 'AT' THEN 'Ativo'
            WHEN a.ind_status = 'PE' THEN 'Pendente'
            ELSE a.ind_status
        END AS ind_status,
        st_area(st_transform(a.geom, 97823))/10000 AS area_imovel_ha,
        (st_area(st_transform(a.geom, 97823))/10000) / c.modulofiscal_ha AS modulos_fiscais,
        CASE
            WHEN (st_area(st_transform(a.geom, 97823))/10000) / c.modulofiscal_ha <= 4 THEN 'Pequeno'
            WHEN (st_area(st_transform(a.geom, 97823))/10000) / c.modulofiscal_ha <= 15 THEN 'Médio'
            ELSE 'Grande'
        END AS tamanho_calculado,
        CASE
            WHEN a.mod_fiscal <= 4 THEN 'Pequeno'
            WHEN a.mod_fiscal <= 15 THEN 'Médio'
            ELSE 'Grande'
        END AS tamanho_informado,
        a.geom
    FROM car_analisar a
    LEFT JOIN {IBGE_SCHEMA}.{MUNICIPIOS_TABLE} b
        ON ST_Intersects(a.geom, b.geom)
    LEFT JOIN {INCRA_SCHEMA}.{MODULOS_FISCAIS_TABLE} c
        ON b.cd_mun::int = c.codibge 
    ORDER BY 
        a.cod_imovel, st_area(st_transform(st_intersection(a.geom, b.geom), 97823)) DESC
)
SELECT
    a.cod_imovel, a.ind_status, a.nm_mun AS municipio,
    CASE 
        WHEN RIGHT(LEFT(a.cod_imovel, 10), 7) = a.cd_mun THEN 'Sim'
        ELSE 'Não'
    END AS municipio_correto, 
    round(a.area_imovel_ha::NUMERIC, 2) AS area_imovel_ha, 
    round(a.modulos_fiscais::NUMERIC, 2) AS modulos_fiscais,
    a.tamanho_calculado, a.tamanho_informado, 
    st_intersection(a.geom, b.geom) AS geom,
    b.regiao,
    round((st_area(st_transform(st_intersection(a.geom, b.geom), 97823))/10000)::numeric, 2) AS area_ha,
    round(((st_area(st_transform(st_intersection(a.geom, b.geom), 97823))/10000) / area_imovel_ha)*100) AS porcentagem_regiao
FROM municipio a
LEFT JOIN {IBGE_SCHEMA}.{AMAZONIA_LEGAL_TABLE} b
    ON st_intersects(a.geom, b.geom)    
GROUP BY 
    a.cod_imovel, a.ind_status, a.nm_mun, a.cd_mun, 
    a.area_imovel_ha, a.modulos_fiscais, a.tamanho_calculado, 
    a.tamanho_informado, a.geom, b.geom, b.regiao;
""")

df_auditoria = pd.read_sql_query(query_auditoria, engine)
df = df_auditoria.drop(columns=['geom'])  # Removendo a coluna de geometria para exibição tabular   

# ============================================================================
# Definição do Dicionário de Metadados 
# ============================================================================

metadados_territorial_dict = {
    "Campo": [
        "cod_imovel", "ind_status", "municipio", "municipio_correto", 
        "area_imovel_ha", "modulos_fiscais", "tamanho_calculado", 
        "tamanho_informado", "geom", "regiao", "area_ha", "porcentagem_regiao"
    ],
    "Descrição": [
        "Código do Imóvel no sistema",
        "Indicador do status do imóvel (SU: Suspenso, CA: Cancelado, AT: Ativo, PE: Pendente)",
        "Município de maior sobreposição com o imóvel",
        "Indicador se município declarado é o de maior sobreposição (Auditoria de localização)",
        "Área total em hectares do imóvel",
        "Quantidade de módulos fiscais calculados no imóvel (baseado no INCRA)",
        "Tamanho do imóvel calculado (P: Pequeno, M: Médio, G: Grande)",
        "Tamanho do imóvel declarado pelo proprietário (P, M, G)",
        "Geometria do Imóvel",
        "Região em que o imóvel sobrepõe (Dentro ou fora da Amazônia Legal)",
        "Área em hectares da porção do imóvel sobreposta com a região",
        "Porcentagem da área do imóvel sobreposta com a região"
    ]
}

df_meta_territorial = pd.DataFrame(metadados_territorial_dict)

# --- EXIBIÇÃO ---

print("--- RESULTADO DA ANÁLISE ---")
display(df.head().style.set_table_styles(estilo_centralizado).hide(axis='index'))

print("\n--- DICIONÁRIO DE METADADOS ---")
display(df_meta_territorial.style.set_table_styles(estilo_centralizado).hide(axis='index'))

--- RESULTADO DA ANÁLISE ---


cod_imovel,ind_status,municipio,municipio_correto,area_imovel_ha,modulos_fiscais,tamanho_calculado,tamanho_informado,regiao,area_ha,porcentagem_regiao
PI-2210102-299FE936E65A4FBE8BADF2C526B91D52,Pendente,São José do Peixe,Sim,8934.490000,127.640000,Grande,Grande,Cerrado,6796.570000,76.000000
PI-2210102-299FE936E65A4FBE8BADF2C526B91D52,Pendente,São José do Peixe,Sim,8934.490000,127.640000,Grande,Grande,Fora da Amazônia Legal,2137.880000,24.000000



--- DICIONÁRIO DE METADADOS ---


Campo,Descrição
cod_imovel,Código do Imóvel no sistema
ind_status,"Indicador do status do imóvel (SU: Suspenso, CA: Cancelado, AT: Ativo, PE: Pendente)"
municipio,Município de maior sobreposição com o imóvel
municipio_correto,Indicador se município declarado é o de maior sobreposição (Auditoria de localização)
area_imovel_ha,Área total em hectares do imóvel
modulos_fiscais,Quantidade de módulos fiscais calculados no imóvel (baseado no INCRA)
tamanho_calculado,"Tamanho do imóvel calculado (P: Pequeno, M: Médio, G: Grande)"
tamanho_informado,"Tamanho do imóvel declarado pelo proprietário (P, M, G)"
geom,Geometria do Imóvel
regiao,Região em que o imóvel sobrepõe (Dentro ou fora da Amazônia Legal)


<hr style="border:1px solid #000000;">

# Visualizar Imóvel e Regiões de Contato

<hr style="border:1px solid #000000;">

Gera um arquivo HTML com a visualização do imóvel e as regiões que toca (Amazônia Legal).


In [ ]:
# ============================================================================
# Processamento Geográfico e Visualização - Auditoria do Imóvel (df_auditoria)
# ============================================================================

# Função para converter WKB do PostGIS para Geometria Shapely
def converter_wkb(wkb_data):
    if wkb_data is None:
        return None
    try:
        # Se for string/hex do PostGIS
        if isinstance(wkb_data, str):
            return wkb.loads(binascii.unhexlify(wkb_data))
        # Se já for bytes
        return wkb.loads(wkb_data)
    except:
        return None

# Tratamento de Geometria de forma segura (Idempotente)
if 'geom' in df_auditoria.columns:
    # Busca o primeiro valor não nulo para verificar o tipo
    dados_validos = df_auditoria['geom'].dropna()
    primeiro_valor = dados_validos.iloc[0] if not dados_validos.empty else None
    
    # Só converte se o dado existir e ainda não for um objeto do Shapely
    if primeiro_valor is not None and not isinstance(primeiro_valor, BaseGeometry):
        df_auditoria['geom'] = df_auditoria['geom'].apply(converter_wkb)

# Mostra os dados sem a coluna geom para ficar organizado
display(df_auditoria.drop(columns=['geom']))

# Converter o DataFrame tratado em GeoDataFrame para o Folium usar
gdf_auditoria = gpd.GeoDataFrame(df_auditoria, geometry='geom', crs="EPSG:4326")

# 1. Dicionário de cores por região
cores_regioes = {
    'Amazônia Legal': '#1E90FF',           # Azul
    'Cerrado na Amazônia Legal': '#FFD700', # Dourado/Amarelo
    'Cerrado': '#FFA500',                  # Laranja
    'Fora da Amazônia Legal': '#8FBC8F'    # Verde Mar Aberto
}

# ============================================================================
# 2. Calcular o centróide 
# ============================================================================
centroid_a = gdf_auditoria.to_crs(epsg=5880).geometry.centroid.to_crs(epsg=4326).iloc[0]

# 3. Inicializar o mapa Folium com super zoom ajustado (Sem fundo cinza)
m_a = folium.Map(location=[centroid_a.y, centroid_a.x], zoom_start=13, max_zoom=22, control_scale=True)

# Camada de Satélite com limitador nativo do servidor
folium.TileLayer(
    'https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}', 
    attr='Google', 
    name='Google Satellite',
    max_native_zoom=18,  # Limite físico do Google
    max_zoom=22          # Permite esticar o pixel para conferência de limites
).add_to(m_a)

# ============================================================================
# 4. MAPEAMENTO EM HARDCODE PARA MODIFICAR OS TÍTULOS NO HOVER DO MOUSE
# ============================================================================
colunas_atributos_a = [
    'cod_imovel', 'ind_status', 'municipio', 'municipio_correto', 
    'area_imovel_ha', 'modulos_fiscais', 'tamanho_calculado', 
    'tamanho_informado', 'regiao', 'area_ha', 'porcentagem_regiao'
]

# Filtra apenas o que de fato veio na query do df_auditoria para não dar KeyError
colunas_validas_a = [col for col in colunas_atributos_a if col in df_auditoria.columns]

# Mude os títulos textuais dentro das strings abaixo para alterar a legenda do mouse
aliases_mapeamento_a = {
    'cod_imovel': 'Código Imóvel:',
    'ind_status': 'Status do Imóvel:',
    'municipio': 'Município:',
    'municipio_correto': 'Município Correto:',
    'area_imovel_ha': 'Área do Imóvel (ha):',
    'modulos_fiscais': 'Módulos Fiscais:',
    'tamanho_calculado': 'Tamanho Calculado:',
    'tamanho_informado': 'Tamanho Declarado:',
    'regiao': 'Região Enquadrada:',
    'area_ha': 'Área (ha):',
    'porcentagem_regiao': 'Porcentagem na Região (%):'
}
aliases_validos_a = [aliases_mapeamento_a[col] for col in colunas_validas_a]

# 5. Adicionar GeoJson com lógica de cores e Tooltip Avançado
folium.GeoJson(
    gdf_auditoria,
    name='Área do Imóvel (por Região)',
    style_function=lambda feature: {
        'fillColor': cores_regioes.get(feature['properties']['regiao'], 'gray'), 
        'color': 'white',
        'weight': 2,
        'fillOpacity': 0.6
    },
    tooltip=folium.GeoJsonTooltip(
        fields=colunas_validas_a,
        aliases=aliases_validos_a,
        localize=True,
        sticky=True,
        labels=True,
        style="""
            background-color: #F8F9FA;
            border: 2px solid #1A1A1A;
            border-radius: 4px;
            box-shadow: 4px 4px rgba(0,0,0,0.15);
            color: #1A1A1A;
            font-family: 'Courier New', monospace;
            font-size: 12px;
            padding: 12px;
            white-space: nowrap;
        """
    )
).add_to(m_a)

# 6. Adicionar controles e salvar
folium.LayerControl().add_to(m_a)

nome_arquivo_auditoria = 'imovel_regioes.html'
m_a.save(nome_arquivo_auditoria)

webbrowser.open(nome_arquivo_auditoria)

,cod_imovel,ind_status,municipio,municipio_correto,area_imovel_ha,modulos_fiscais,tamanho_calculado,tamanho_informado,regiao,area_ha,porcentagem_regiao
0,PI-2210102-299FE936E65A4FBE8BADF2C526B91D52,Pendente,São José do Peixe,Sim,8934.49,127.64,Grande,Grande,Cerrado,6796.57,76.0
1,PI-2210102-299FE936E65A4FBE8BADF2C526B91D52,Pendente,São José do Peixe,Sim,8934.49,127.64,Grande,Grande,Fora da Amazônia Legal,2137.88,24.0


True

<hr style="border:1px solid #000000;">

# 4. Analisar APP Declarada

<hr style="border:1px solid #000000;">

Função que permite analisar a APP Total e a Área Consolidada declaradas.

In [14]:
# ============================================================================
# Definição de Schemas e Tabelas CAR
# ============================================================================

CAR_SCHEMA = 'car'
SICAR_TABLE = 'es_pi_area_imovel_sicar'
APP_TABLE = 'es_pi_apps_sicar'
CONSOLIDADA_TABLE = 'es_pi_area_consolidada_sicar'

# ============================================================================
# Definição de Schemas e Tabelas IBGE
# ============================================================================

IBGE_SCHEMA = 'ibge'
MUNICIPIOS_TABLE = 'es_pi_municipios_ibge_2025'

# ============================================================================
# Definição de Schemas e Tabelas INCRA
# ============================================================================

INCRA_SCHEMA = 'incra'
MODULOS_FISCAIS_TABLE = 'pa_br_modulosfiscais_incra'

# ============================================================================
# Identificadores (COD_IMOVEL)
# ============================================================================

COD_IMOVEL = 'PI-2210003-B03533D53FE243BCB31069D7097D59CB' # APP sem desconto
# COD_IMOVEL = 'PI-2204303-C537B8423457434386A8003BB50CF0E4' # Possui redução 61-A
# COD_IMOVEL = 'PI-2200400-A377B1177E914D8792BCE4AB99F39875' # Grande, depende de ZEE
# COD_IMOVEL = 'PI-2204303-59FFAE72C88648FAA013FAE36A658BA0' # Sem APP declarada

# ============================================================================
# CONSULTA SQL PARA ANÁLISE DO IMÓVEL SELECIONADO
# ============================================================================

query_app = text(f"""
WITH car_analisar AS (
    SELECT * FROM {CAR_SCHEMA}.{SICAR_TABLE}
    WHERE cod_imovel = '{COD_IMOVEL}'
)
SELECT
    a.cod_imovel, 
    CASE
        WHEN st_area(st_transform(a.geom,97823))/10000 / c.modulofiscal_ha <= 4 THEN 'Pequeno'
        WHEN st_area(st_transform(a.geom,97823))/10000 / c.modulofiscal_ha <= 15 THEN 'Médio'
        ELSE 'Grande'
    END AS tamanho,
    a.geom AS imovel,
    CASE 
        WHEN d.cod_imovel IS NULL THEN 'Não'
        ELSE 'Sim'
    END AS possui_app_declarada,
    st_union(d.geom) AS app,
    CASE 
        WHEN st_area(st_transform(a.geom,97823))/10000 / c.modulofiscal_ha <= 4 THEN 'Sim'
        ELSE 'Não (Depende do ZEE)'
    END AS apto_para_61a,
    CASE 
        WHEN e.cod_imovel IS NULL THEN 'Não'
        ELSE 'Sim'
    END AS possui_area_consolidada,
    st_union(e.geom) AS area_consolidada
FROM car_analisar a
JOIN {IBGE_SCHEMA}.{MUNICIPIOS_TABLE} b
    ON ST_Intersects(a.geom, b.geom)
JOIN {INCRA_SCHEMA}.{MODULOS_FISCAIS_TABLE} c
    ON b.cd_mun::int = c.codibge 
LEFT JOIN {CAR_SCHEMA}.{APP_TABLE} d
    USING (cod_imovel)
LEFT JOIN {CAR_SCHEMA}.{CONSOLIDADA_TABLE} e
    USING (cod_imovel)
GROUP BY 
    a.cod_imovel, a.geom, c.modulofiscal_ha,
    d.cod_imovel, e.cod_imovel;
""")

df_app = pd.read_sql_query(query_app, engine)
df = df_app.drop(columns=['imovel', 'app', 'area_consolidada'])  # Removendo colunas de geometria para exibição tabular

# ============================================================================
# Definição do Dicionário de Metadados 
# ============================================================================

metadados_app_consolidada_dict = {
    "Campo": [
        "cod_imovel", "tamanho", "imovel", "possui_app_declarada", 
        "app", "apto_para_61a", "possui_area_consolidada", "area_consolidada"
    ],
    "Descrição": [
        "Código do Imóvel no sistema",
        "Tamanho do imóvel calculado (P: Pequeno, M: Médio, G: Grande)",
        "Geometria do Imóvel",
        "Indicador se possui APP declarada",
        "Geometria da APP Total declarada",
        "Indicador se imóvel está apto a ter reduções segundo o artigo 61-A",
        "Indicador se possui Área Consolidada declarada",
        "Geometria da Área Consolidada declarada"
    ]
}

df_meta_app_consolidada = pd.DataFrame(metadados_app_consolidada_dict)

# --- EXIBIÇÃO ---

print("--- RESULTADO DA ANÁLISE ---")
display(df.head().style.set_table_styles(estilo_centralizado).hide(axis='index'))

print("\n--- DICIONÁRIO DE METADADOS ---")
display(df_meta_app_consolidada.style.set_table_styles(estilo_centralizado).hide(axis='index'))

--- RESULTADO DA ANÁLISE ---


cod_imovel,tamanho,possui_app_declarada,apto_para_61a,possui_area_consolidada
PI-2210003-B03533D53FE243BCB31069D7097D59CB,Pequeno,Sim,Sim,Não



--- DICIONÁRIO DE METADADOS ---


Campo,Descrição
cod_imovel,Código do Imóvel no sistema
tamanho,"Tamanho do imóvel calculado (P: Pequeno, M: Médio, G: Grande)"
imovel,Geometria do Imóvel
possui_app_declarada,Indicador se possui APP declarada
app,Geometria da APP Total declarada
apto_para_61a,Indicador se imóvel está apto a ter reduções segundo o artigo 61-A
possui_area_consolidada,Indicador se possui Área Consolidada declarada
area_consolidada,Geometria da Área Consolidada declarada


<hr style="border:1px solid #000000;">

# Visualizar Imóvel, APP e Área Consolidada

<hr style="border:1px solid #000000;">

Gera um arquivo HTML com a visualização do imóvel, APP Total e a Área Consolidada declaradas.

In [ ]:
# ============================================================================
# Processamento Geográfico e Visualização - Análise de APP (df_app)
# ============================================================================

# Função para converter WKB do PostGIS para Geometria Shapely
def converter_wkb(wkb_data):
    if wkb_data is None:
        return None
    try:
        # Se for string/hex do PostGIS
        if isinstance(wkb_data, str):
            return wkb.loads(binascii.unhexlify(wkb_data))
        # Se já for bytes
        return wkb.loads(wkb_data)
    except:
        return None

# 1. Tratamento de Geometria de forma segura (Idempotente)
colunas_geom_app = ['imovel', 'app', 'area_consolidada']

for col in colunas_geom_app:
    if col in df_app.columns:
        # Busca o primeiro valor não nulo para verificar o tipo
        dados_validos = df_app[col].dropna()
        primeiro_valor = dados_validos.iloc[0] if not dados_validos.empty else None
        
        # Só converte se o dado existir e ainda não for um objeto do Shapely (Geometria)
        if primeiro_valor is not None and not isinstance(primeiro_valor, BaseGeometry):
            df_app[col] = df_app[col].apply(converter_wkb)

# Exibir tabela de auditoria sem as geometrias
display(df_app.drop(columns=colunas_geom_app))

# 2. Configuração do GeoDataFrame base para o centroide
gdf_base_app = gpd.GeoDataFrame(df_app, geometry='imovel', crs="EPSG:4326")
centroid_app = gdf_base_app.to_crs(epsg=5880).geometry.centroid.to_crs(epsg=4326).iloc[0]

# 3. Inicializar o mapa Folium com super zoom ajustado (Sem fundo cinza)
m_app = folium.Map(location=[centroid_app.y, centroid_app.x], zoom_start=15, max_zoom=22, control_scale=True)

# Camada de Satélite com limitador nativo do servidor
folium.TileLayer(
    'https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}', 
    attr='Google', 
    name='Google Satellite',
    max_native_zoom=18,  # Limite físico do Google
    max_zoom=22          # Permite esticar o pixel para conferência de limites
).add_to(m_app)

# ============================================================================
# 4. MAPEAMENTO DINAMICO DE ATRIBUTOS PARA TOOLTIP (todas as colunas exceto as de geometria)
# ============================================================================
# Seleciona todas as colunas alfanuméricas, ignorando as colunas geométricas passadas na lista
colunas_validas_app = [col for col in df_app.columns if col not in colunas_geom_app]

# Gera rótulos limpos automaticamente (ex: 'apto_para_61a' vira 'Apto Para 61A:')
aliases_validos_app = [f"{col.replace('_', ' ').title()}:" for col in colunas_validas_app]

# 5. Dicionário de Estilização para as camadas do mapa
config_mapa_app = {
    'imovel': {'nome': 'Limite do Imóvel', 'cor': 'white', 'fill': False, 'weight': 3},
    'app': {'nome': 'APP Declarada', 'cor': '#4169E1', 'fill': True, 'opacity': 0.5},
    'area_consolidada': {'nome': 'Uso Consolidado', 'cor': '#FFD700', 'fill': True, 'opacity': 0.4}
}

# 6. Adicionar cada feição ao mapa com os dados totais atrelados
for col, config in config_mapa_app.items():
    if col in df_app.columns and df_app[col].iloc[0] is not None:
        
        # Cria o GeoDataFrame associando a geometria ativa com TODAS as colunas tabulares de diagnóstico
        temp_gdf = gpd.GeoDataFrame(df_app[colunas_validas_app + [col]], geometry=col, crs="EPSG:4326")
        
        # Configuração do Tooltip Flutuante Dinâmico e Completo
        tooltip_feicao = folium.GeoJsonTooltip(
            fields=colunas_validas_app,
            aliases=aliases_validos_app,
            localize=True,
            sticky=True,
            labels=True,
            style="""
                background-color: #F8F9FA;
                border: 2px solid #1A1A1A;
                border-radius: 4px;
                box-shadow: 4px 4px rgba(0,0,0,0.15);
                color: #1A1A1A;
                font-family: 'Courier New', monospace;
                font-size: 12px;
                padding: 12px;
                white-space: nowrap;
            """
        )
        
        folium.GeoJson(
            temp_gdf,
            name=config['nome'],
            style_function=lambda x, c=config: {
                'fillColor': c['cor'] if c['fill'] else 'transparent',
                'color': c['cor'],
                'weight': c.get('weight', 2),
                'fillOpacity': c.get('opacity', 0.5)
            },
            tooltip=tooltip_feicao
        ).add_to(m_app)

# 7. Adicionar controles e salvar
folium.LayerControl().add_to(m_app)

nome_arquivo_app = 'imovel_app_declarada.html'
m_app.save(nome_arquivo_app)

webbrowser.open(nome_arquivo_app)

,cod_imovel,tamanho,possui_app_declarada,apto_para_61a,possui_area_consolidada
0,PI-2210003-B03533D53FE243BCB31069D7097D59CB,Pequeno,Sim,Sim,Não


True

<hr style="border:1px solid #000000;">

# 5. Estimar Déficit de APP

<hr style="border:1px solid #000000;">

Estimativa do Déficit de APP no imóvel baseado somente nas informações auto declaradas.

In [20]:
# ============================================================================
# Definição de Schemas e Tabelas CAR
# ============================================================================

CAR_SCHEMA = 'car'
SICAR_TABLE = 'es_pi_area_imovel_sicar'
APP_TABLE = 'es_pi_apps_sicar'
VEGETACAO_TABLE = 'es_pi_vegetacao_nativa_sicar'

# ============================================================================
# Definição de Schemas e Tabelas IBGE
# ============================================================================

IBGE_SCHEMA = 'ibge'
MUNICIPIOS_TABLE = 'es_pi_municipios_ibge_2025'

# ============================================================================
# Definição de Schemas e Tabelas INCRA
# ============================================================================

INCRA_SCHEMA = 'incra'
MODULOS_FISCAIS_TABLE = 'pa_br_modulosfiscais_incra'

# ============================================================================
# Identificadores (COD_IMOVEL)
# ============================================================================

# COD_IMOVEL = 'PI-2210003-B03533D53FE243BCB31069D7097D59CB' # APP com déficit
COD_IMOVEL = 'PI-2204303-C537B8423457434386A8003BB50CF0E4' # APP com déficit na faixa mínima
# COD_IMOVEL = 'PI-2200400-A377B1177E914D8792BCE4AB99F39875'
# COD_IMOVEL = 'PI-2204303-59FFAE72C88648FAA013FAE36A658BA0' # Sem déficits

# ============================================================================
# CONSULTA SQL PARA ANÁLISE DO IMÓVEL SELECIONADO
# ============================================================================

query_deficit = text(f"""
WITH car_analisar AS (
    SELECT * FROM {CAR_SCHEMA}.{SICAR_TABLE}
    WHERE cod_imovel = '{COD_IMOVEL}'
),
app AS (    
    SELECT
        a.cod_imovel, 
        CASE
            WHEN st_area(st_transform(a.geom,97823))/10000 / c.modulofiscal_ha <= 4 THEN 'Pequeno'
            WHEN st_area(st_transform(a.geom,97823))/10000 / c.modulofiscal_ha <= 15 THEN 'Médio'
            ELSE 'Grande'
        END AS tamanho,
        a.geom AS imovel,
        st_union(d.geom) AS app_declarada,
        CASE 
            WHEN st_area(st_transform(a.geom,97823))/10000 / c.modulofiscal_ha <= 4 THEN 'Sim'
            ELSE 'Não (Depende do ZEE)'
        END AS apto_para_61a,
        st_union(d.geom) FILTER (WHERE d.cod_tema LIKE 'APP_ESCADINHA%') AS app_minima,
        st_union(e.geom) AS remanescente_vegetacao
    FROM car_analisar a
    JOIN {IBGE_SCHEMA}.{MUNICIPIOS_TABLE} b
        ON ST_Intersects(a.geom, b.geom)
    JOIN {INCRA_SCHEMA}.{MODULOS_FISCAIS_TABLE} c
        ON b.cd_mun::int = c.codibge 
    LEFT JOIN {CAR_SCHEMA}.{APP_TABLE} d
        USING (cod_imovel)
    LEFT JOIN {CAR_SCHEMA}.{VEGETACAO_TABLE} e
        USING (cod_imovel)
    GROUP BY 
        a.cod_imovel, a.geom, c.modulofiscal_ha, d.cod_imovel
)
SELECT 
    *,
    COALESCE(
        st_area(st_transform(st_difference(app_declarada, remanescente_vegetacao), 97823))/10000,
        st_area(st_transform(app_declarada, 97823))/10000
    ) AS deficit_app_total_ha,
    COALESCE(
        st_area(st_transform(st_difference(app_minima, remanescente_vegetacao), 97823))/10000,
        st_area(st_transform(app_minima, 97823))/10000
    ) AS deficit_app_minima_ha
FROM app;
""")

df_deficit = pd.read_sql_query(query_deficit, engine)
df = df_deficit.drop(columns=['imovel', 'app_declarada', 'app_minima', 'remanescente_vegetacao'])  # Removendo colunas de geometria para exibição tabular

# ============================================================================
# Definição do Dicionário de Metadados 
# ============================================================================

metadados_deficit_app_dict = {
    "Campo": [
        "cod_imovel", "tamanho", "imovel", "app_declarada", "apto_para_61a",
        "apto_minima", "remanescente_vegetacao", "deficit_app_total_ha", "deficit_app_minima_ha"
    ],
    "Descrição": [
        "Código do Imóvel no sistema",
        "Tamanho do imóvel calculado (P, M, G)",
        "Geometria do Imóvel",
        "Geometria da APP Total declarada",
        "Indicador se imóvel está apto a ter reduções segundo o artigo 61-A",
        "Geometria da APP Mínima declarada (Regra da Escadinha)",
        "Geometria do Remanescente de Vegetação Nativa declarado",
        "Área em hectares da APP Total sem cobertura de Vegetação Nativa",
        "Área em hectares da APP Mínima sem cobertura de Vegetação Nativa"
    ]
}

df_meta_deficit_app = pd.DataFrame(metadados_deficit_app_dict)

# --- EXIBIÇÃO ---
print("--- RESULTADO DA ANÁLISE ---")
display(df.head().style.set_table_styles(estilo_centralizado).hide(axis='index'))

print("\n--- DICIONÁRIO DE METADADOS ---")
display(df_meta_deficit_app.style.set_table_styles(estilo_centralizado).hide(axis='index'))

--- RESULTADO DA ANÁLISE ---


cod_imovel,tamanho,apto_para_61a,deficit_app_total_ha,deficit_app_minima_ha
PI-2204303-C537B8423457434386A8003BB50CF0E4,Pequeno,Sim,7.876930,3.880466



--- DICIONÁRIO DE METADADOS ---


Campo,Descrição
cod_imovel,Código do Imóvel no sistema
tamanho,"Tamanho do imóvel calculado (P, M, G)"
imovel,Geometria do Imóvel
app_declarada,Geometria da APP Total declarada
apto_para_61a,Indicador se imóvel está apto a ter reduções segundo o artigo 61-A
apto_minima,Geometria da APP Mínima declarada (Regra da Escadinha)
remanescente_vegetacao,Geometria do Remanescente de Vegetação Nativa declarado
deficit_app_total_ha,Área em hectares da APP Total sem cobertura de Vegetação Nativa
deficit_app_minima_ha,Área em hectares da APP Mínima sem cobertura de Vegetação Nativa


<hr style="border:1px solid #000000;">

# Visualizar Imóvel, APP e Remanescentes de Vegetação Nativa

<hr style="border:1px solid #000000;">

Gera um arquivo HTML com a visualização do imóvel, APP Total, APP Mínima e Remanescentes de Vegetação Nativa declarado

In [ ]:
# ============================================================================
# Processamento Geográfico e Visualização - Análise de Déficit (df_deficit)
# ============================================================================

# Função para converter WKB do PostGIS para Geometria Shapely
def converter_wkb(wkb_data):
    if wkb_data is None:
        return None
    try:
        # Se for string/hex do PostGIS
        if isinstance(wkb_data, str):
            return wkb.loads(binascii.unhexlify(wkb_data))
        # Se já for bytes
        return wkb.loads(wkb_data)
    except:
        return None

# 1. Tratamento de Geometria de forma segura (Idempotente)
colunas_geom_deficit = ['imovel', 'app_declarada', 'app_minima', 'remanescente_vegetacao']

for col in colunas_geom_deficit:
    if col in df_deficit.columns:
        # Busca o primeiro valor não nulo para verificar o tipo
        dados_validos = df_deficit[col].dropna()
        primeiro_valor = dados_validos.iloc[0] if not dados_validos.empty else None
        
        # Só converte se o dado existir e ainda não for um objeto do Shapely
        if primeiro_valor is not None and not isinstance(primeiro_valor, BaseGeometry):
            df_deficit[col] = df_deficit[col].apply(converter_wkb)

# 2. Exibição da Tabela Corrigida sem as geometrias
df_display = df_deficit.drop(columns=colunas_geom_deficit).copy()

display(df_display.style.format({
    'deficit_app_total_ha': '{:.4f} ha',
    'deficit_app_minima_ha': '{:.4f} ha'
}, na_rep='-').background_gradient(cmap='YlOrRd', subset=['deficit_app_total_ha']))

# 3. Configuração do GeoDataFrame base para o centroide
gdf_base_def = gpd.GeoDataFrame(df_deficit, geometry='imovel', crs="EPSG:4326")
centroid_def = gdf_base_def.to_crs(epsg=5880).geometry.centroid.to_crs(epsg=4326).iloc[0]

# 4. Inicializar o mapa Folium com super zoom ajustado (Sem fundo cinza)
m_def = folium.Map(location=[centroid_def.y, centroid_def.x], zoom_start=15, max_zoom=22, control_scale=True)

# Camada de Satélite com limitador nativo do servidor
folium.TileLayer(
    'https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}', 
    attr='Google', 
    name='Google Satellite',
    max_native_zoom=18,  # Limite físico do Google
    max_zoom=22          # Permite esticar o pixel para conferência de limites
).add_to(m_def)

# ============================================================================
# 5. MAPEAMENTO EM HARDCODE PARA MODIFICAR OS TÍTULOS NO HOVER DO MOUSE
# ============================================================================
colunas_atributos_def = [
    'cod_imovel', 'tamanho', 'apto_para_61a', 
    'deficit_app_total_ha', 'deficit_app_minima_ha'
]

# Filtra apenas o que de fato veio na query do df_deficit para não dar KeyError
colunas_validas_def = [col for col in colunas_atributos_def if col in df_deficit.columns]

# Mude os títulos textuais dentro das strings abaixo para alterar a legenda do mouse
aliases_mapeamento_def = {
    'cod_imovel': 'Código Imóvel:',
    'tamanho': 'Tamanho do Imóvel:',
    'apto_para_61a': 'Apto Art. 61-A?:',
    'deficit_app_total_ha': 'Déficit APP Total (ha):',
    'deficit_app_minima_ha': 'Déficit APP Mínima (ha):'
}
aliases_validos_def = [aliases_mapeamento_def[col] for col in colunas_validas_def]

# 6. Dicionário de Estilização das camadas
config_mapa_def = {
    'imovel':                 {'nome': 'Limite do Imóvel',    'cor': 'white',   'fill': False, 'weight': 3},
    'remanescente_vegetacao': {'nome': 'Vegetação Nativa',   'cor': '#228B22', 'fill': True,  'opacity': 0.4},
    'app_declarada':          {'nome': 'APP Total (Lei)',     'cor': '#4169E1', 'fill': True,  'opacity': 0.3},
    'app_minima':             {'nome': 'APP Mínima (61-A)',  'cor': '#FF4500', 'fill': True,  'opacity': 0.5}
}

# 7. Adicionar cada feição ao mapa injetando todos os dados de diagnóstico no Hover
for col, config in config_mapa_def.items():
    if col in df_deficit.columns and df_deficit[col].iloc[0] is not None:
        
        # Cria o GeoDataFrame associando a geometria ativa com TODAS as colunas tabulares
        temp_gdf = gpd.GeoDataFrame(df_deficit[colunas_validas_def + [col]], geometry=col, crs="EPSG:4326")
        
        # Configuração do Tooltip Flutuante Dinâmico e Customizado
        tooltip_feicao = folium.GeoJsonTooltip(
            fields=colunas_validas_def,
            aliases=aliases_validos_def,
            localize=True,
            sticky=True,
            labels=True,
            style="""
                background-color: #F8F9FA;
                border: 2px solid #1A1A1A;
                border-radius: 4px;
                box-shadow: 4px 4px rgba(0,0,0,0.15);
                color: #1A1A1A;
                font-family: 'Courier New', monospace;
                font-size: 12px;
                padding: 12px;
                white-space: nowrap;
            """
        )
        
        folium.GeoJson(
            temp_gdf,
            name=config['nome'],
            style_function=lambda x, c=config: {
                'fillColor': c['cor'] if c['fill'] else 'transparent',
                'color': c['cor'],
                'weight': c.get('weight', 2),
                'fillOpacity': c.get('opacity', 0.5)
            },
            tooltip=tooltip_feicao
        ).add_to(m_def)

# 8. Adicionar controles e salvar
folium.LayerControl().add_to(m_def)

nome_arquivo_deficit = 'imovel_app_remanescente.html'
m_def.save(nome_arquivo_deficit)

webbrowser.open(nome_arquivo_deficit)

,cod_imovel,tamanho,apto_para_61a,deficit_app_total_ha,deficit_app_minima_ha
0,PI-2204303-C537B8423457434386A8003BB50CF0E4,Pequeno,Sim,7.8769 ha,3.8805 ha


True

<hr style="border:1px solid #000000;">

# 6. Analisar Reserva Legal Declarada

<hr style="border:1px solid #000000;">

Função que permite analisar a Reserva Legal e o correto dimensionamento para cada região do imóvel.

In [22]:
# ============================================================================
# Definição de Schemas e Tabelas CAR
# ============================================================================

CAR_SCHEMA = 'car'
SICAR_TABLE = 'es_pi_area_imovel_sicar'
RL_TABLE = 'es_pi_reserva_legal_sicar'

# ============================================================================
# Definição de Schemas e Tabelas IBGE
# ============================================================================

IBGE_SCHEMA = 'ibge'
AMAZONIA_LEGAL_TABLE = 'pa_br_amazonia_legal'

# ============================================================================
# Identificadores (COD_IMOVEL)
# ============================================================================

COD_IMOVEL = 'PI-2204303-C537B8423457434386A8003BB50CF0E4' # RL bem dimensionada
# COD_IMOVEL = 'PI-2211209-B3866F03F500495F9EFC79FF754E6A1D' # Amazônia Legal
# COD_IMOVEL = 'PI-2210102-299FE936E65A4FBE8BADF2C526B91D52' # Déficit de RL

# ============================================================================
# CONSULTA SQL PARA ANÁLISE DO IMÓVEL SELECIONADO
# ============================================================================

query_rl = text(f"""
WITH car_analisar AS (
    SELECT * FROM {CAR_SCHEMA}.{SICAR_TABLE}
    WHERE cod_imovel = '{COD_IMOVEL}'
),
imovel AS (
    SELECT
        a.cod_imovel,
        a.cod_estado,
        CASE 
            WHEN b.regiao IS NULL THEN 'Fora da Amazônia Legal'
            ELSE b.regiao
        END AS regiao,
        st_area(st_transform(st_intersection(a.geom, b.geom),97823))/10000 AS area_regiao_ha,
        ((st_area(st_transform(st_intersection(a.geom, b.geom),97823))/10000)
            / (st_area(st_transform(a.geom,97823))/10000)*100)
        AS porcentagem_do_imovel,
        st_area(st_transform(a.geom,97823))/10000 AS area_imovel_ha
    FROM car_analisar a
    LEFT JOIN {IBGE_SCHEMA}.{AMAZONIA_LEGAL_TABLE} b
        ON st_intersects(a.geom, b.geom)
),
rl_exigida AS (
    SELECT 
        *,
        CASE
            WHEN regiao = 'Fora da Amazônia Legal' THEN '20%'
            WHEN regiao = 'Cerrado' THEN CASE WHEN cod_estado = 'PI' THEN '30%' ELSE '20%' END
            WHEN regiao = 'Amazônia Legal' THEN '80%'
            WHEN regiao = 'Cerrado na Amazônia Legal' THEN '35%'
        END AS regra_rl_exigida,
        CASE
            WHEN regiao = 'Fora da Amazônia Legal' THEN (area_imovel_ha*(porcentagem_do_imovel/100))*0.2
            WHEN regiao = 'Cerrado' THEN CASE WHEN cod_estado = 'PI' THEN (area_imovel_ha*(porcentagem_do_imovel/100))*0.3 ELSE (area_imovel_ha*(porcentagem_do_imovel/100))*0.2 END
            WHEN regiao = 'Amazônia Legal' THEN (area_imovel_ha*(porcentagem_do_imovel/100))*0.8
            WHEN regiao = 'Cerrado na Amazônia Legal' THEN COALESCE((area_imovel_ha*(porcentagem_do_imovel/100))*0.35, area_imovel_ha*0.35)
        END AS rl_exigida_ha
    FROM imovel
),
rl_declarada AS (
    SELECT 
        a.cod_imovel, 
        CASE 
            WHEN b.regiao IS NULL THEN 'Fora da Amazônia Legal'
            ELSE b.regiao
        END AS regiao,
        st_union(st_intersection(a.geom,b.geom)) AS rl,
        (st_area(st_transform(st_intersection(a.geom,b.geom),97823))/10000) AS area_rl_regiao_ha
    FROM {CAR_SCHEMA}.{RL_TABLE} a
    LEFT JOIN {IBGE_SCHEMA}.{AMAZONIA_LEGAL_TABLE} b
            ON st_intersects(a.geom, b.geom)
    LEFT JOIN car_analisar c
        USING (cod_imovel)
    WHERE c.cod_imovel IS NOT null
    GROUP BY a.cod_imovel, a.geom, b.geom, b.regiao
)
SELECT 
    a.*,
    b.rl, b.area_rl_regiao_ha,
    CASE
        WHEN rl_exigida_ha - area_rl_regiao_ha <= 0 THEN 'Dimensões suficientes'
        WHEN rl_exigida_ha - area_rl_regiao_ha > 0 THEN 'Dimensões insuficientes'
        WHEN b.cod_imovel IS NULL THEN 'Não declarada'
    END AS reserva_declarada,
    CASE
        WHEN rl_exigida_ha - area_rl_regiao_ha <= 0 THEN NULL 
        WHEN rl_exigida_ha - area_rl_regiao_ha > 0 THEN rl_exigida_ha - area_rl_regiao_ha
        WHEN b.cod_imovel IS NULL THEN rl_exigida_ha
    END AS diferenca_rl_ha,
    c.geom AS imovel
FROM rl_exigida a
LEFT JOIN rl_declarada b USING (cod_imovel, regiao)
LEFT JOIN car_analisar c USING (cod_imovel);
""")

df_rl = pd.read_sql_query(query_rl, engine)
df = df_rl.drop(columns=['rl', 'imovel'])  # Removendo colunas de geometria para exibição tabular

# ============================================================================
# Definição do Dicionário de Metadados
# ============================================================================

metadados_rl_dict = {
    "Campo": [
        "cod_imovel", "cod_estado", "regiao", "area_regiao_ha", 
        "porcentagem_do_imovel", "area_imovel_ha", "regra_rl_exigida", 
        "rl_exigida_ha", "rl", "area_rl_regiao_ha", "reserva_declarada", 
        "diferenca_rl_ha", "imovel"
    ],
    "Descrição": [
        "Código do Imóvel no sistema",
        "Sigla do estado do imóvel",
        "Região em que o imóvel sobrepõe (Dentro ou fora da Amazônia Legal)",
        "Área em hectares da porção do imóvel sobreposta com a região",
        "Porcentagem da área do imóvel sobreposta com a região",
        "Área total em hectares do imóvel",
        "Indicador da regra aplicada na porção do imóvel sobre a região",
        "Área em hectares da Reserva Legal exigida na porção do imóvel sobre a região",
        "Geometria da Reserva Legal declarada",
        "Área em hectares da Reserva Legal declarada na porção do imóvel sobre a região",
        "Indicador sobre as dimensões da Reserva Legal declarada (Suficientes ou Insuficientes)",
        "Área em hectares adicional de Reserva Legal necessária para cumprir o exigido",
        "Geometria do Imóvel"
    ]
}

df_meta_rl = pd.DataFrame(metadados_rl_dict)

# --- EXIBIÇÃO ---
print("--- RESULTADO DA ANÁLISE ---")
display(df.head().style.set_table_styles(estilo_centralizado).hide(axis='index'))

print("\n--- DICIONÁRIO DE METADADOS ---")
display(df_meta_rl.style.set_table_styles(estilo_centralizado).hide(axis='index'))

--- RESULTADO DA ANÁLISE ---


cod_imovel,cod_estado,regiao,area_regiao_ha,porcentagem_do_imovel,area_imovel_ha,regra_rl_exigida,rl_exigida_ha,area_rl_regiao_ha,reserva_declarada,diferenca_rl_ha
PI-2204303-C537B8423457434386A8003BB50CF0E4,PI,Fora da Amazônia Legal,142.169819,100.000000,142.169819,20%,28.433964,29.378752,Dimensões suficientes,None



--- DICIONÁRIO DE METADADOS ---


Campo,Descrição
cod_imovel,Código do Imóvel no sistema
cod_estado,Sigla do estado do imóvel
regiao,Região em que o imóvel sobrepõe (Dentro ou fora da Amazônia Legal)
area_regiao_ha,Área em hectares da porção do imóvel sobreposta com a região
porcentagem_do_imovel,Porcentagem da área do imóvel sobreposta com a região
area_imovel_ha,Área total em hectares do imóvel
regra_rl_exigida,Indicador da regra aplicada na porção do imóvel sobre a região
rl_exigida_ha,Área em hectares da Reserva Legal exigida na porção do imóvel sobre a região
rl,Geometria da Reserva Legal declarada
area_rl_regiao_ha,Área em hectares da Reserva Legal declarada na porção do imóvel sobre a região


<hr style="border:1px solid #000000;">

# Visualizar Imóvel e Reserva Legal

<hr style="border:1px solid #000000;">

Gera um arquivo HTML com a visualização do imóvel e  Reserva Legal declarada.

In [ ]:
# ============================================================================
# Processamento Geográfico e Visualização - Reserva Legal (df_rl)
# ============================================================================

# Função para converter WKB do PostGIS para Geometria Shapely
def converter_wkb(wkb_data):
    if wkb_data is None:
        return None
    try:
        # Se for string/hex do PostGIS
        if isinstance(wkb_data, str):
            return wkb.loads(binascii.unhexlify(wkb_data))
        # Se já for bytes
        return wkb.loads(wkb_data)
    except:
        return None

# 1. Tratamento de Geometria de forma segura (Idempotente)
colunas_geom_rl = ['imovel', 'rl']

for col in colunas_geom_rl:
    if col in df_rl.columns:
        # Busca o primeiro valor não nulo para verificar o tipo
        dados_validos = df_rl[col].dropna()
        primeiro_valor = dados_validos.iloc[0] if not dados_validos.empty else None
        
        # Só converte se o dado existir e ainda não for um objeto do Shapely
        if primeiro_valor is not None and not isinstance(primeiro_valor, BaseGeometry):
            df_rl[col] = df_rl[col].apply(converter_wkb)

# 2. Exibição da Tabela Corrigida sem as geometrias
df_display = df_rl.drop(columns=colunas_geom_rl).copy()

display(df_display.style.format({
    'rl_exigida_ha': '{:.2f} ha',
    'area_rl_regiao_ha': '{:.2f} ha',
    'diferenca_rl_ha': '{:.2f} ha'
}, na_rep='Saldo OK').map(
    lambda x: 'color: red; font-weight: bold' if isinstance(x, (int, float)) and x > 0 else '', 
    subset=['diferenca_rl_ha']
))

# 3. Configuração do GeoDataFrame base para o centroide
gdf_base_rl = gpd.GeoDataFrame(df_rl, geometry='imovel', crs="EPSG:4326")
centroid_rl = gdf_base_rl.to_crs(epsg=5880).geometry.centroid.to_crs(epsg=4326).iloc[0]

# 4. Inicializar o mapa Folium com super zoom ajustado (Sem fundo cinza)
m_rl = folium.Map(location=[centroid_rl.y, centroid_rl.x], zoom_start=13, max_zoom=22, control_scale=True)

# Camada de Satélite com limitador nativo do servidor
folium.TileLayer(
    'https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}', 
    attr='Google', 
    name='Google Satellite',
    max_native_zoom=18,  # Limite físico do Google
    max_zoom=22          # Permite esticar o pixel para conferência de limites
).add_to(m_rl)

# ============================================================================
# 5. MAPEAMENTO EM HARDCODE PARA MODIFICAR OS TÍTULOS NO HOVER DO MOUSE
# ============================================================================
colunas_atributos_rl = [
    'cod_imovel', 'cod_estado', 'regiao', 'area_regiao_ha', 
    'porcentagem_do_imovel', 'area_imovel_ha', 'regra_rl_exigida', 
    'rl_exigida_ha', 'area_rl_regiao_ha', 'reserva_declarada', 
    'diferenca_rl_ha'
]

# Filtra apenas o que de fato veio na query do df_rl para não dar KeyError
colunas_validas_rl = [col for col in colunas_atributos_rl if col in df_rl.columns]

# Mude os títulos textuais dentro das strings abaixo para alterar a legenda do mouse
aliases_mapeamento_rl = {
    'cod_imovel': 'Código Imóvel:',
    'cod_estado': 'Código Estado:',
    'regiao': 'Região Enquadrada:',
    'area_regiao_ha': 'Área Região (ha):',
    'porcentagem_do_imovel': 'Porcentagem do Imóvel (%):',
    'area_imovel_ha': 'Área Imóvel (ha):',
    'regra_rl_exigida': 'Regra RL Exigida (%):',
    'rl_exigida_ha': 'RL Exigida (ha):',
    'area_rl_regiao_ha': 'Área de RL Região (ha):',
    'reserva_declarada': 'Reserva Declarada:',
    'diferenca_rl_ha': 'Diferença RL (ha):'
}
aliases_validos_rl = [aliases_mapeamento_rl[col] for col in colunas_validas_rl]

# 6. Dicionário de Estilização das camadas
config_mapa_rl = {
    'imovel': {'nome': 'Limite do Imóvel', 'cor': 'white',   'fill': False, 'weight': 3},
    'rl':     {'nome': 'Reserva Legal (RL)', 'cor': '#006400', 'fill': True,  'opacity': 0.6}
}

# 7. Adicionar cada feição ao mapa injetando o DataFrame de atributos completo
for col, config in config_mapa_rl.items():
    if col in df_rl.columns and df_rl[col].iloc[0] is not None:
        
        # Cria o GeoDataFrame associando a geometria ativa com TODAS as colunas tabulares de análise
        temp_gdf = gpd.GeoDataFrame(df_rl[colunas_validas_rl + [col]], geometry=col, crs="EPSG:4326")
        
        # Configuração do Tooltip Flutuante Dinâmico e Customizado
        tooltip_feicao = folium.GeoJsonTooltip(
            fields=colunas_validas_rl,
            aliases=aliases_validos_rl,
            localize=True,
            sticky=True,
            labels=True,
            style="""
                background-color: #F8F9FA;
                border: 2px solid #1A1A1A;
                border-radius: 4px;
                box-shadow: 4px 4px rgba(0,0,0,0.15);
                color: #1A1A1A;
                font-family: 'Courier New', monospace;
                font-size: 12px;
                padding: 12px;
                white-space: nowrap;
            """
        )
        
        folium.GeoJson(
            temp_gdf,
            name=config['nome'],
            style_function=lambda x, c=config: {
                'fillColor': c['cor'] if c['fill'] else 'transparent',
                'color': c['cor'],
                'weight': c.get('weight', 2),
                'fillOpacity': c.get('opacity', 0.5) if c['fill'] else 0.0
            },
            tooltip=tooltip_feicao
        ).add_to(m_rl)

# 8. Adicionar controles e salvar
folium.LayerControl().add_to(m_rl)

nome_arquivo_rl = 'imovel_rl_declarada.html'
m_rl.save(nome_arquivo_rl)

webbrowser.open(nome_arquivo_rl)

,cod_imovel,cod_estado,regiao,area_regiao_ha,porcentagem_do_imovel,area_imovel_ha,regra_rl_exigida,rl_exigida_ha,area_rl_regiao_ha,reserva_declarada,diferenca_rl_ha
0,PI-2204303-C537B8423457434386A8003BB50CF0E4,PI,Fora da Amazônia Legal,142.169819,100.000000,142.169819,20%,28.43 ha,29.38 ha,Dimensões suficientes,Saldo OK


True

<hr style="border:1px solid #000000;">

# 7. Estimar Déficit de Reserva Legal

<hr style="border:1px solid #000000;">

Estimativa do Déficit de RL no imóvel baseado somente nas informações auto declaradas.

In [20]:
# ============================================================================
# Definição de Schemas e Tabelas CAR
# ============================================================================

CAR_SCHEMA = 'car'
SICAR_TABLE = 'es_pi_area_imovel_sicar'
RL_TABLE = 'es_pi_reserva_legal_sicar'
VN_TABLE = 'es_pi_vegetacao_nativa_sicar'

# ============================================================================
# Definição de Schemas e Tabelas IBGE
# ============================================================================

IBGE_SCHEMA = 'ibge'
AMAZONIA_LEGAL_TABLE = 'pa_br_amazonia_legal'

# ============================================================================
# Identificadores (COD_IMOVEL)
# ============================================================================

COD_IMOVEL = 'PI-2210003-B03533D53FE243BCB31069D7097D59CB' # Déficit grande de RL
# COD_IMOVEL = 'PI-2204303-C537B8423457434386A8003BB50CF0E4' # Sem VN declarada
# COD_IMOVEL = 'PI-2204303-59FFAE72C88648FAA013FAE36A658BA0' # VN fora da RL
# COD_IMOVEL = 'PI-2210607-2AF64E7F67E84DE5B7634D6D01EEFEA3' # Sem déficit

# ============================================================================
# CONSULTA SQL PARA ANÁLISE DO IMÓVEL SELECIONADO
# ============================================================================

query_deficit_rl = text(f"""
WITH car_analisar AS (
    SELECT * FROM {CAR_SCHEMA}.{SICAR_TABLE}
    WHERE cod_imovel = '{COD_IMOVEL}'
),
imovel AS (
    SELECT
        a.cod_imovel,
        a.cod_estado,
        CASE 
            WHEN b.regiao IS NULL THEN 'Fora da Amazônia Legal'
            ELSE b.regiao
        END AS regiao,
        ((st_area(st_transform(st_intersection(a.geom,b.geom),97823))/10000) / (st_area(st_transform(a.geom,97823))/10000)*100)
            AS porcentagem_do_imovel,
        st_area(st_transform(a.geom,97823))/10000 AS area_imovel_ha
    FROM car_analisar a
    LEFT JOIN {IBGE_SCHEMA}.{AMAZONIA_LEGAL_TABLE} b
        ON st_intersects(a.geom, b.geom)
),
rl_exigida AS (
    SELECT 
        *,
        CASE
            WHEN regiao = 'Fora da Amazônia Legal' THEN (area_imovel_ha*(porcentagem_do_imovel/100))*0.2
            WHEN regiao = 'Cerrado' THEN
                CASE WHEN cod_estado = 'PI' THEN (area_imovel_ha*(porcentagem_do_imovel/100))*0.3
                ELSE (area_imovel_ha*(porcentagem_do_imovel/100))*0.2 END
            WHEN regiao = 'Amazônia Legal' THEN (area_imovel_ha*(porcentagem_do_imovel/100))*0.8
            WHEN regiao = 'Cerrado na Amazônia Legal' THEN COALESCE((area_imovel_ha*(porcentagem_do_imovel/100))*0.35, area_imovel_ha*0.35)
        END AS rl_exigida_ha
    FROM imovel
),
rl_declarada AS (
    SELECT 
        a.cod_imovel, 
        CASE 
            WHEN b.regiao IS NULL THEN 'Fora da Amazônia Legal'
            ELSE b.regiao
        END AS regiao,
        st_union(st_intersection(a.geom,b.geom)) AS rl,
        (st_area(st_transform(st_intersection(a.geom,b.geom),97823))/10000) AS area_rl_regiao_ha
    FROM {CAR_SCHEMA}.{RL_TABLE} a
    LEFT JOIN {IBGE_SCHEMA}.{AMAZONIA_LEGAL_TABLE} b
            ON st_intersects(a.geom, b.geom)
    LEFT JOIN car_analisar c USING (cod_imovel)
    WHERE c.cod_imovel IS NOT NULL
    GROUP BY a.cod_imovel, a.geom, b.geom, b.regiao
),
vn_declarada AS (
    SELECT  
        a.cod_imovel, 
        CASE 
            WHEN b.regiao IS NULL THEN 'Fora da Amazônia Legal'
            ELSE b.regiao
        END AS regiao,
        (st_area(st_transform(st_intersection(a.geom,b.rl),97823))/10000) AS area_vn_rl_ha
    FROM {CAR_SCHEMA}.{VN_TABLE} a
    LEFT JOIN rl_declarada b ON st_intersects(a.geom, b.rl)
    LEFT JOIN car_analisar c ON a.cod_imovel = c.cod_imovel
    WHERE c.cod_imovel IS NOT NULL
    GROUP BY a.cod_imovel, a.geom, b.rl, b.regiao
),
dados_finais AS (
    SELECT 
        a.cod_imovel, 
        st_union(b.rl) AS rl_declarada,
        sum(rl_exigida_ha) AS rl_exigida_ha,
        sum(CASE
            WHEN c.area_vn_rl_ha - rl_exigida_ha < 0 THEN rl_exigida_ha - c.area_vn_rl_ha
            WHEN c.area_vn_rl_ha - rl_exigida_ha >= 0 THEN 0
            ELSE rl_exigida_ha
        END) AS deficit_rl_ha,
        st_area(st_transform(COALESCE(st_difference(e.geom, st_union(rl)), e.geom), 97823))/10000 AS vn_fora_rl_ha,
        e.geom AS vn_declarada
    FROM rl_exigida a
    LEFT JOIN rl_declarada b USING (cod_imovel, regiao)
    LEFT JOIN vn_declarada c USING (cod_imovel, regiao)
    LEFT JOIN {CAR_SCHEMA}.{VN_TABLE} e USING (cod_imovel)
    GROUP BY a.cod_imovel, e.geom
)
SELECT
    cod_imovel, rl_declarada, 
    CASE
        WHEN deficit_rl_ha > 0 THEN 'Sim'
        ELSE 'Não'
    END AS possui_deficit_rl,
    vn_declarada, vn_fora_rl_ha, deficit_rl_ha,
    CASE
        WHEN deficit_rl_ha = 0 THEN 0
        ELSE round(((deficit_rl_ha*100)/rl_exigida_ha))
    END AS perc_rl_sem_vn,
    b.geom AS imovel
FROM dados_finais
LEFT JOIN car_analisar b USING (cod_imovel);
""")

df_def_rl = pd.read_sql_query(query_deficit_rl, engine)
df = df_def_rl.drop(columns=['rl_declarada', 'vn_declarada', 'imovel'])  # Removendo colunas de geometria para exibição tabular

# ============================================================================
# Definição do Dicionário de Metadados
# ============================================================================

metadados_deficit_rl_vn_dict = {
    "Campo": [
        "cod_imovel", "rl_declarada", "possui_deficit_rl", "vn_declarada", 
        "vn_fora_rl_ha", "deficit_rl_ha", "perc_rl_sem_vn", "imovel"
    ],
    "Descrição": [
        "Código do Imóvel no sistema",
        "Geometria da Reserva Legal declarada",
        "Indicador se possui déficit em Reserva Legal",
        "Geometria do Remanescente de Vegetação Nativa declarado",
        "Área em hectares do Remanescente de Vegetação Nativa fora da Reserva Legal declarada",
        "Área em hectares da Reserva Legal exigida sem cobertura de Vegetação Nativa",
        "Porcentagem da área da Reserva Legal exigida sem cobertura de Vegetação Nativa",
        "Geometria do Imóvel"
    ]
}

df_meta_deficit_rl_vn = pd.DataFrame(metadados_deficit_rl_vn_dict)

# --- EXIBIÇÃO ---
print("--- RESULTADO DA ANÁLISE ---")
display(df.head().style.set_table_styles(estilo_centralizado).hide(axis='index'))

print("\n--- DICIONÁRIO DE METADADOS ---")
display(df_meta_deficit_rl_vn.style.set_table_styles(estilo_centralizado).hide(axis='index'))

--- RESULTADO DA ANÁLISE ---


cod_imovel,possui_deficit_rl,vn_fora_rl_ha,deficit_rl_ha,perc_rl_sem_vn
PI-2210003-B03533D53FE243BCB31069D7097D59CB,Sim,6.572785,2.821053,98.000000



--- DICIONÁRIO DE METADADOS ---


Campo,Descrição
cod_imovel,Código do Imóvel no sistema
rl_declarada,Geometria da Reserva Legal declarada
possui_deficit_rl,Indicador se possui déficit em Reserva Legal
vn_declarada,Geometria do Remanescente de Vegetação Nativa declarado
vn_fora_rl_ha,Área em hectares do Remanescente de Vegetação Nativa fora da Reserva Legal declarada
deficit_rl_ha,Área em hectares da Reserva Legal exigida sem cobertura de Vegetação Nativa
perc_rl_sem_vn,Porcentagem da área da Reserva Legal exigida sem cobertura de Vegetação Nativa
imovel,Geometria do Imóvel


<hr style="border:1px solid #000000;">

# Visualizar Imóvel, Reserva Legal e Remanescentes de Vegetação Nativa

<hr style="border:1px solid #000000;">

Gera um arquivo HTML com a visualização do imóvel, RL e Remanescentes de Vegetação Nativa declarados.

In [ ]:
# ============================================================================
# Processamento Geográfico e Visualização - Déficit de RL (df_def_rl)
# ============================================================================

# Função para converter WKB do PostGIS para Geometria Shapely
def converter_wkb(wkb_data):
    if wkb_data is None:
        return None
    try:
        # Se for string/hex do PostGIS
        if isinstance(wkb_data, str):
            return wkb.loads(binascii.unhexlify(wkb_data))
        # Se já for bytes
        return wkb.loads(wkb_data)
    except:
        return None

# 1. Tratamento de Geometria de forma segura (Idempotente)
colunas_geom_def = ['imovel', 'rl_declarada', 'vn_declarada']

for col in colunas_geom_def:
    if col in df_def_rl.columns:
        # Busca o primeiro valor não nulo para verificar o tipo
        dados_validos = df_def_rl[col].dropna()
        primeiro_valor = dados_validos.iloc[0] if not dados_validos.empty else None
        
        # Só converte se o dado existir e ainda não for um objeto do Shapely
        if primeiro_valor is not None and not isinstance(primeiro_valor, BaseGeometry):
            df_def_rl[col] = df_def_rl[col].apply(converter_wkb)

# 2. Exibição da Tabela (Corrigida para Pandas 2.1+) sem as geometrias
df_disp = df_def_rl.drop(columns=colunas_geom_def).copy()

display(df_disp.style.format({
    'deficit_rl_ha': '{:.2f} ha',
    'vn_fora_rl_ha': '{:.2f} ha',
    'perc_rl_sem_vn': '{} %'
}, na_rep='0.00').map(
    lambda x: 'color: red; font-weight: bold' if isinstance(x, (int, float)) and x > 0 else '', 
    subset=['deficit_rl_ha', 'perc_rl_sem_vn']
))

# 3. Configuração do GeoDataFrame base para o centroide
gdf_base_def = gpd.GeoDataFrame(df_def_rl, geometry='imovel', crs="EPSG:4326")
centroid_def = gdf_base_def.to_crs(epsg=5880).geometry.centroid.to_crs(epsg=4326).iloc[0]

# 4. Inicializar o mapa Folium com super zoom ajustado (Sem fundo cinza)
m_def = folium.Map(location=[centroid_def.y, centroid_def.x], zoom_start=14, max_zoom=22, control_scale=True)

# Camada de Satélite com limitador nativo do servidor
folium.TileLayer(
    'https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}', 
    attr='Google', 
    name='Google Satellite',
    max_native_zoom=18,  # Limite físico do Google
    max_zoom=22          # Permite esticar o pixel para conferência de limites
).add_to(m_def)

# ============================================================================
# 5. MAPEAMENTO EM HARDCODE PARA MODIFICAR OS TÍTULOS NO HOVER DO MOUSE
# ============================================================================
colunas_atributos_def = [
    'cod_imovel', 'possui_deficit_rl', 'vn_fora_rl_ha', 
    'deficit_rl_ha', 'perc_rl_sem_vn'
]

# Filtra apenas o que de fato veio na query do df_def_rl para não dar KeyError
colunas_validas_def = [col for col in colunas_atributos_def if col in df_def_rl.columns]

# Mude os títulos textuais dentro das strings abaixo para alterar a legenda do mouse
aliases_mapeamento_def = {
    'cod_imovel': 'Código Imóvel:',
    'possui_deficit_rl': 'Possui Déficit RL?:',
    'vn_fora_rl_ha': 'VN Fora RL (ha):',
    'deficit_rl_ha': 'Déficit RL (ha):',
    'perc_rl_sem_vn': 'Perc RL sem VN (%):'
}
aliases_validos_def = [aliases_mapeamento_def[col] for col in colunas_validas_def]

# 6. Dicionário de Estilização das camadas
config_mapa_def = {
    'imovel':       {'nome': 'Limite do Imóvel', 'cor': 'white',   'fill': False, 'weight': 3},
    'vn_declarada': {'nome': 'Vegetação Nativa', 'cor': '#228B22', 'fill': True,  'opacity': 0.4},
    'rl_declarada': {'nome': 'Reserva Legal',   'cor': '#FF8C00', 'fill': True,  'opacity': 0.3}
}

# 7. Adicionar cada feição ao mapa injetando o DataFrame de atributos completo
for col, config in config_mapa_def.items():
    if col in df_def_rl.columns and df_def_rl[col].iloc[0] is not None:
        
        # Cria o GeoDataFrame associando a geometria ativa com TODAS as colunas tabulares de análise
        temp_gdf = gpd.GeoDataFrame(df_def_rl[colunas_validas_def + [col]], geometry=col, crs="EPSG:4326")
        
        # Configuração do Tooltip Flutuante Dinâmico e Customizado
        tooltip_feicao = folium.GeoJsonTooltip(
            fields=colunas_validas_def,
            aliases=aliases_validos_def,
            localize=True,
            sticky=True,
            labels=True,
            style="""
                background-color: #F8F9FA;
                border: 2px solid #1A1A1A;
                border-radius: 4px;
                box-shadow: 4px 4px rgba(0,0,0,0.15);
                color: #1A1A1A;
                font-family: 'Courier New', monospace;
                font-size: 12px;
                padding: 12px;
                white-space: nowrap;
            """
        )
        
        folium.GeoJson(
            temp_gdf,
            name=config['nome'],
            style_function=lambda x, c=config: {
                'fillColor': c['cor'] if c['fill'] else 'transparent',
                'color': c['cor'],
                'weight': c.get('weight', 2),
                'fillOpacity': c.get('opacity', 0.5) if c['fill'] else 0.0
            },
            tooltip=tooltip_feicao
        ).add_to(m_def)

# 8. Adicionar controles e salvar
folium.LayerControl().add_to(m_def)

nome_arquivo_def = 'imovel_rl_remanescente.html'
m_def.save(nome_arquivo_def)

webbrowser.open(nome_arquivo_def)

,cod_imovel,possui_deficit_rl,vn_fora_rl_ha,deficit_rl_ha,perc_rl_sem_vn
0,PI-2210003-B03533D53FE243BCB31069D7097D59CB,Sim,6.57 ha,2.82 ha,98.0 %


True

<hr style="border:1px solid #000000;">

# Referências

<hr style="border:1px solid #000000;">

- [SiCAR](https://consultapublica.car.gov.br/publico/imoveis/index)
- [Lei nº 12.651/2012 - Lei de Proteção da Vegetação Nativa](https://www.planalto.gov.br/ccivil_03/_ato2011-2014/2012/lei/l12651.html)
- [Modulo Fiscal - INCRA](https://www.gov.br/incra/pt-br/assuntos/governanca-fundiaria/modulo-fiscal)
- [Amazônia Legal - IBGE](https://www.ibge.gov.br/geociencias/cartas-e-mapas/mapas-regionais/15819-amazonia-legal.html)
- [Municípios Brasil - IBGE](https://www.ibge.gov.br/geociencias/organizacao-do-territorio/malhas-territoriais/15774-malhas.html)